In [ ]:
import os
import sys
import warnings
import time
import json
from natsort import natsorted
from pathlib import PureWindowsPath, PurePosixPath
import pickle
import numpy as np
import xarray as xr
import pandas as pd
import math 
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.pyplot import figure
from matplotlib.patches import Patch, Rectangle
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
import seaborn as sns
import cmasher as cmr

from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks, peak_widths
import scipy.stats as stats
from scipy.stats import skew, median_abs_deviation
import scikit_posthocs as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf

sys.path.append('../utils') 
from utils_tfc import TFC_proto
from utils_tfc import open_minian, xrconcat_recursive, map_ts
from utils_plot import (
    set_pub_style, get_asterisks, save_metadata_json, 
    lighten_color, add_stat_annotation_two_sided)

#General parameters
dpath_cal_all = r'../../data/11.Post_TFC-20' # The directory for TFC-20
dpath_cal_supp = r'../../data/' # The directory for others (TFC-60, TFC-5, TFC-0)
bin_width = 200  # ms
fs = int(1000/bin_width)

colors_anatomy = ['#A6761D', '#845ec2', '#97cebf'] # '#A6761D' gold '#f28482' Soft Coral/Pink, Vibrant Amethyst, Deep Violet/Midnight
colors_beh_i = ['#4091cf', '#e1703c'] # Blue, red
colors_beh_e = ['#4091cf', '#8cba54'] # Blue, green
#plot
dir_output = r'../output_figures'
os.makedirs(dir_output, exist_ok=True)
dir_fig = 'Fig5'
dpath_plot = os.path.join(dir_output, dir_fig)
if not os.path.exists(dpath_plot):
    os.makedirs(dpath_plot)   

## 3.0 supp-TFC-60, 5, 0--Basic Mean Z score plot with different trace durations (60s, 5s, 0s)(in CA1)

In [ ]:
# TFC-60, TFC-5, TFC-0 plot
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)
flag_raw = 1 
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 40  # 
height_mm = 30 #  

trace_groups = ['13.Post_TFC-60', '14.Post_TFC-5', '15.Post_TFC-0']
trace_durs = [60, 5, 0]
for idx_g, trace_key in enumerate(trace_groups):
    dpath_cal_all_2 = f'{dpath_cal_supp}{trace_key}'
    trace_du = trace_durs[idx_g]

    base_du = 3 # 10s
    post_du = 40 #post shock  use 20s
    bins_trial_start = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
    bins_trial_end = int((20+20+trace_du+3+ post_du)*fs) # post-shock 17s
    
    cal_mean_trials_animal = dict()
    cal_mean_2_trials_animal = dict()
    for i in range(group_size):    
        dpath_cal_group = os.path.join(dpath_cal_all_2, group_name[i])
        dpath_test = os.path.join(dpath_cal_group, test_algori_data)
        print(dpath_test) 
        if flag_raw == 1:
            Sig_bin_trial_base = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_base.nc"))    # The base is always 20s   
            Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))        
        bins_with_base = Sig_bin_trial_base.bins.size + Sig_bin_trial.bins.size
        Sig_bin_trial = xr.concat([Sig_bin_trial_base, Sig_bin_trial], dim='bins').assign_coords({'bins': range(bins_with_base)})#.drop_vars("session")  
        Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].sel(session='session0_condi')           
        #Extract data
        cal_mean_trials_animal[group_keys[i]] = Sig_bin_trial.mean(dim='trials').sel(bins=range(bins_trial_start, bins_trial_end)).values
        cal_mean_2_trials_animal[group_keys[i]] = Sig_bin_trial.sel(trials=range(0,2), bins=range(bins_trial_start, bins_trial_end)).mean(dim='trials').values
  
    # 01. ALl 6 trials
    epochs = {'Base':3 , 'CS': 20, 'Trace': trace_du, 'US': 3, 'P_US1': 3, 'P_US2': 6}
    plot_trial_population_activity_with_timeline(cal_mean_trials_animal, width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, f'sup_03_0_{idx_g+1}_All 6 trials-Mean Z score-animal-wise-TFC-{trace_du}')

    # 02. First 2 trials
    plot_trial_population_activity_with_timeline(cal_mean_2_trials_animal, width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, f'sup_03_0_{idx_g+4}_First 2 trials-Mean Z score-animal-wise-TFC-{trace_du}')
print('All finished************') 

In [ ]:
def plot_trial_population_activity_with_timeline(cal_trial, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    Includes a floating timeline schematic mapped to shading colors/alphas, 
    with a continuous baseline spanning CS to US.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING & TIMELINE SCHEMATIC ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15)}      # Postw: dark Gray        
    
    current_time = t_start
    timeline_events = {}
    
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        
        # Track start and end coordinates for the schematic
        timeline_events[ep_name] = (current_time, current_time + ep_dur)
        
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # Floating Timeline Schematic (Skipped if Trace == 20)
    trace_dur = epochs.get('Trace', None)
    if trace_dur is not None and trace_dur != 20:
        # Increase space to PETH plot (1.15 pushes it nicely above the top spine)
        y_base = 1.05 
        if 'Trace' in timeline_events:
            trace_s, trace_e = timeline_events['Trace']
            trace_center = (trace_s + trace_e) / 2

            ax.plot([trace_s, trace_e], [y_base, y_base], transform=ax.get_xaxis_transform(),
                    color='black', lw=0.5, clip_on=False)
            # Place text just inside the gap above the baseline. 
            # If trace_dur == 0, it perfectly centers on the CS/US border intersection.
            ax.text(trace_center, y_base + 0.015, f"{int(trace_dur)} s", 
                    transform=ax.get_xaxis_transform(), ha='center', va='bottom', 
                    fontsize=6, color='black', clip_on=False)
                    
    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',  labelpad=1)
    ax.set_ylabel('Mean Z-Score', labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 3.1 supp-Basic Mean Z score plot (all cells, animal-wise in CA1)

In [ ]:
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I']
group_size = len(group_name)

flag_raw = 1
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

base_du = 3 # 10s
post_du = 40 #post shock  use 20s
bins_trial_start = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
bins_trial_end = int((20+20+20+3+ post_du)*fs) # post-shock 17s

cal_mean_trials_animal = dict()
cal_mean_2_trials_animal = dict()
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test) 
    if flag_raw == 1:
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))        
    Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].sel(session='session0_condi')           
    #Extract data
    cal_mean_trials_animal[group_keys[i]] = Sig_bin_trial.mean(dim='trials').sel(bins=range(bins_trial_start, bins_trial_end)).values
    cal_mean_2_trials_animal[group_keys[i]] = Sig_bin_trial.sel(trials=range(0,2), bins=range(bins_trial_start, bins_trial_end)).mean(dim='trials').values

width_mm = 40  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

# 1. plot of ALl 6 trials -Mean Z score 
plot_trial_population_activity(cal_mean_trials_animal, 'Mean Z-Score',width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 'sup_03_1_1_All 6 trials-Mean Z score-CA1-animal-wise')
# 2. plot of first 2 trials -Mean Z score 
plot_trial_population_activity(cal_mean_2_trials_animal,'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 'sup_03_1_2_First 2 trials-mean Z score-CA1-animal-wise')

print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.1)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        if (y_label=='Mean Speed (cm/s)') & (i==1):
            ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", linestyle='--', zorder=3)
        else:
            ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
            
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',  labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 3.2 supp-Cell proportions of epoch resp cells

In [ ]:
def extract_data_groups_each_trial(df, group_size, group_keys, resp_keys, trial_idx):
    data_group =[]
    for i, resp_k in enumerate(resp_keys):
        data_trials = []
        for j in range(group_size):
            data_trials.append(df.loc[group_keys[j]][resp_k +'_' +str(trial_idx)].values *100) # to %
        data_group.append(data_trials)
    return data_group

def extract_data_groups_average(df, group_size, group_keys, resp_keys, trial_num):
    data_group =[]
    for i, resp_k in enumerate(resp_keys):
        data_trials = []
        for j in range(group_size):
            tmp_trial = []
            for trial_idx in range(trial_num):
                tmp_trial.append(df.loc[group_keys[j]][resp_k +'_' +str(trial_idx+1)].values *100)
            tmp_trial = np.array(tmp_trial).mean(axis=0)
            data_trials.append(tmp_trial) # to %
        data_group.append(data_trials)
    return data_group
    
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'

ds_all_group_stat =[]
for i in range(group_size):    
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data)
    print(dpath_test)    
    df_stat = pd.read_csv(os.path.join(dpath_test, "Resp_cells_statistics_cal.csv")) 
    ds_all_group_stat.append(df_stat)
    
df_stat_all = pd.concat(ds_all_group_stat, keys=group_keys, names=["group", "row"])
resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']
resp_keys_trials = ['cs_first', 'trace_first', 'us_first', 'p1_us_first', 'p2_us_first']
epoch_names=['CS', 'Trace', 'US', 'pUS-1', 'pUS-2']

width_mm = 40  # 
height_mm = 30 #    
# 1. Plot the average epoch resp cell proportions in the first 2 trials
resp_trials_mean = extract_data_groups_average(df_stat_all, group_size, group_keys, resp_keys, trial_num=2)
plot_epoch_proportion_no_norm(resp_trials_mean, width_mm, height_mm, colors_beh_i, group_keys, epoch_names, dpath_plot,  'sup_03_2_Epoch Resp cell proportions averaged in first 2 trials') 

print('All finished************') 

In [ ]:
def plot_epoch_proportion_no_norm(groups_data, width_mm, height_mm, colors, labels, epoch_names, output_path, title):
    set_pub_style()         
    # 1. Exact millimeter canvas with constrained layout
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')      
    n_epochs = len(epoch_names)
    n_groups = len(labels)
    base_positions = np.arange(1, n_epochs + 1)   
    
    # MODIFIED: Offsets for exactly 2 groups (centers them around the tick)
    offsets = [-0.15, 0.15]
    box_width = 0.15
    jitter_strength = 0.04
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}        
    # --- 1. PRE-FLIGHT CHECK: Normalize Data & Find Global Max ---
    global_max = 0
    epoch_data_normalized = [] 
    
    for ep_idx in range(n_epochs):
        ep_name = epoch_names[ep_idx]
        metadata["Data_Summary"][ep_name] = {}
        
        ep_norm_data = []        
        for grp_idx in range(n_groups):
            group_label = labels[grp_idx]            
            
            # Clean and normalize the data
            d_raw = np.array(groups_data[ep_idx][grp_idx], dtype=float)
            d_clean = d_raw[~np.isnan(d_raw)]
            d_norm = d_clean       
            
            ep_norm_data.append(d_norm)
            
            n_mice = len(d_norm)
            metadata["Data_Summary"][ep_name][group_label] = {
                "N_mice": n_mice,
                "Mean": float(np.mean(d_norm)) if n_mice > 0 else 0,
                "SEM": float(stats.sem(d_norm)) if n_mice > 0 else 0
            }            
            
            if n_mice > 0:
                local_max = np.max(d_norm)
                if local_max > global_max:
                    global_max = local_max
                    
        epoch_data_normalized.append(ep_norm_data)
        
    # 1.3 gives a 30% headroom buffer for double-stacked brackets.
    ax.set_ylim(0, global_max * 1.3) 

    # --- 3. PLOTTING LOOP ---
    for ep_idx in range(n_epochs):
        base_x = base_positions[ep_idx]
        ep_norm_data = epoch_data_normalized[ep_idx]
        
        # A. Plot Box and Scatter
        for grp_idx in range(n_groups):
            d_norm = ep_norm_data[grp_idx]
            if len(d_norm) == 0: continue
            
            x_pos = base_x + offsets[grp_idx]
            color = colors[grp_idx]
            
            # Transparent faces, solid edges
            face_color_rgba = mcolors.to_rgba(color, alpha=0.3)
            
            # Boxplot 
            ax.boxplot(d_norm, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))
            
            # Scatter (Tiny points with white borders)
            x_scatter = x_pos + np.random.uniform(-jitter_strength, jitter_strength, size=len(d_norm))
            ax.scatter(x_scatter, d_norm, s=2.5, color=color, 
                       alpha=1.0, edgecolors='white', linewidth=0.25, zorder=3)

        # B. Add Significance Brackets (MODIFIED FOR 2 GROUPS)
        if len(ep_norm_data) >= 2:
            d_grp1 = ep_norm_data[0]
            d_grp2 = ep_norm_data[1]
            
            local_ep_max = max([np.max(d) if len(d)>0 else 0 for d in ep_norm_data])
            
            x_grp1 = base_x + offsets[0]
            x_grp2 = base_x + offsets[1]
            # Bracket 1: Group 1 vs Group 2 (Single test)
            add_stat_annotation_two_sided(ax, d_grp1, d_grp2, x_grp1, x_grp2, local_ep_max, ttest=0, paired=0)      
            _, p_val = stats.mannwhitneyu(d_grp1, d_grp2, alternative='two-sided')
            metadata["Data_Summary"][epoch_names[ep_idx]]['p_val'] = p_val


    # --- FORMATTING ---
    ax.set_xticks(base_positions)
    ax.set_xticklabels(epoch_names, fontsize=6)
    
    # Consolidated to single line to prevent clipping, with labelpad buffer
    ax.set_ylabel('Cell Proportion (%)', labelpad=0.1) 
    
    # REPLACED hardcoded MultipleLocator with dynamic MaxNLocator
    if global_max > 50:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))   
    else:
        ax.yaxis.set_major_locator(ticker.MultipleLocator(10))  
    # MODIFIED: Dynamically scale legend range to n_groups instead of hardcoded 3
    custom_lines = [Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=4, alpha=1.0) for i in range(n_groups)]
    ax.legend(custom_lines, labels, frameon=False, loc='upper left')    
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Changed from -0.5 to 0.5 to perfectly center the boxes on the canvas
    ax.set_xlim(0.5, n_epochs + 0.5)

    # --- STRICT EXPORTING ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    # Increased w_pad slightly (to 0.05 inches) to give the Y-label physical room to exist
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    #0.05    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 3.3 supp-Mean Z score plot-Epoch resp cells (trace)- animal-wise in CA1 

In [ ]:
def extract_PETH_mean_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp) #[:, bin_s:bin_e].mean(axis=0))        
        if len(data_trials) > 0:
            data_trials = np.concatenate(data_trials, axis=0)
            data_trials = np.nanmean(data_trials, axis=0)[bin_s:bin_e]
            data_group.append(data_trials)
    return np.array(data_group)
    
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 3 # 
post_du = 40 #
bin_s = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
bin_e = int((20+20+20+3+ post_du)*fs) #

width_mm = 40  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

resp_names = ['trace']#  'tone', 'trace',
resp_keys = ['trace']
for idx, resp_name in enumerate(resp_names):
    ds_groups = {}
    for i in range(group_size):    
        dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
        with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
            cal_tmp = pickle.load(f)
        
        ds_groups[group_keys[i]] = extract_PETH_mean_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=6)       
    # 1. plot of ALl 6 trials -Mean Z score 
    plot_trial_population_activity(ds_groups, 'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 
                                   f'sup_03_3_all 6 trials-Mean Z score-CA1-animal-wise_{resp_keys[idx]} resp cells')
print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    cal_trial : dict()--group name: (n_anmials, n_bins)
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.4))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.1 CS cell data plotting-heatmap

In [ ]:
group_name = ['02.CA1-C'] 
group_keys = ['CA1-C' ] 
group_size = len(group_name)

resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']

test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 50  # 
height_mm = 60 #   
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US': 9}
base_du = 20
base_plot_start = ((base_du - epochs['Base'])* fs)
base_plot_end = ((base_du +40 + 40)* fs)
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    dpath_cal_bin = os.path.join(dpath_cal_group, test_algori_data)

    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)    # (n_cells, n_trials)
    
    dict_group_cal = {}
    for key in list(dict_group_response.keys()):
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_cal_bin, f"{key}_Cal_bin_trial.nc"))  
        dict_group_cal[key] = Sig_bin_trial['Sig_each_trial'].values[:, :, base_plot_start:base_plot_end]
    plot_cs_subgroup_heatmap(dict_group_response, dict_group_cal, epochs, fs, width_mm, height_mm, dpath_plot, 
                            f'04_1_cs cell activity heatmap-trial 1-{group_keys[i]}')
print('All finished************')      

In [ ]:
def plot_cs_subgroup_heatmap(dict_group_response, dict_group_cal, epochs, fs, width_mm, height_mm, output_path, title):
    """
    Plots a highly optimized, subgroup-sorted heatmap of raw Z-scored calcium traces
    for Trial 1 CS cells pooled across all animals in the group.
    
    Subgroups (Top to Bottom):
    1. CS & US & pUS (Full Integration)
    2. CS & pUS (Binding Only)
    3. CS & US (Triggering Only)
    4. CS Only (Sensory Encoding)
    """
    set_pub_style()    
    # 1. EXTRACT AND POOL DATA FOR TRIAL 1
    t_idx = 0 
    
    pooled_calcium = []
    pooled_flags = []
    
    for animal in dict_group_response.keys():
        flags = dict_group_response[animal]
        cal_data = dict_group_cal[animal] # Shape: (trials, cells, bins)
        
        # Get Trial 1 calcium traces for this animal
        cal_t1 = cal_data[t_idx] 
        
        # Get Trial 1 flags (Transpose so shape is (cells,))
        cs_t1 = np.array(flags['cs']).astype(bool).T[t_idx]
        us_t1 = np.array(flags['us']).astype(bool).T[t_idx]
        pus1_t1 = np.array(flags['pus_1']).astype(bool).T[t_idx]
        pus2_t1 = np.array(flags['pus_2']).astype(bool).T[t_idx]
        
        pus_t1 = pus1_t1 | pus2_t1
        
        # only care about cells that are CS-responsive
        cs_indices = np.where(cs_t1)[0]
        
        for idx in cs_indices:
            pooled_calcium.append(cal_t1[idx])
            pooled_flags.append({
                'us': us_t1[idx],
                'pus': pus_t1[idx]
            })

    pooled_calcium = np.array(pooled_calcium)
    if len(pooled_calcium) == 0:
        raise ValueError("No CS responsive cells found in Trial 1.")

    # 2. ASSIGN SUBGROUPS
    subgroup_ids = np.zeros(len(pooled_flags), dtype=int)
    for i, flag in enumerate(pooled_flags):
        if flag['us'] and flag['pus']:
            subgroup_ids[i] = 0
        elif flag['us'] and not flag['pus']:
            subgroup_ids[i] = 1
        elif not flag['us'] and flag['pus']:
            subgroup_ids[i] = 2
        else:
            subgroup_ids[i] = 3

    # 3. INTERNAL PEAK SORTING
    t_base_bins = int(epochs.get('Base', 3) * fs)
    t_cs_bins = int(epochs.get('CS', 20) * fs)
    
    cs_start = t_base_bins
    cs_end = t_base_bins + t_cs_bins
    
    sorted_calcium = []
    subgroup_boundaries = [0] 
    
    for g_id in range(4):
        group_indices = np.where(subgroup_ids == g_id)[0]
        if len(group_indices) == 0:
            continue
            
        group_cal = pooled_calcium[group_indices]
        
        cs_epoch_data = group_cal[:, cs_start:cs_end]
        peak_times = np.argmax(cs_epoch_data, axis=1)
        
        sort_order = np.argsort(peak_times)
        sorted_group_cal = group_cal[sort_order]
        
        sorted_calcium.append(sorted_group_cal)
        subgroup_boundaries.append(subgroup_boundaries[-1] + len(group_indices))

    final_matrix = np.vstack(sorted_calcium)

    # 4. PLOTTING THE HEATMAP
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Calculate time boundaries in seconds to map the X-axis accurately
    base_sec = epochs.get('Base', 3)
    total_sec = final_matrix.shape[1] / fs
    x_start = -base_sec
    x_end = total_sec - base_sec
    n_cells = final_matrix.shape[0]

    # Map the array mathematically to [Time Start, Time End, Bottom Cell, Top Cell]
    im = ax.imshow(final_matrix, aspect='auto', cmap='coolwarm', 
                   vmin=-3, vmax=3, interpolation='nearest', #vmin=-0.5, vmax=3.5
                   extent=[x_start, x_end, n_cells, 0])             
    # 5. AESTHETICS & EPOCH ANNOTATIONS
    # Draw vertical lines for epoch transitions using REAL time (seconds)
    current_time_sec = -base_sec
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time_sec += ep_dur
            continue
        ax.axvline(current_time_sec, color='white', linestyle='--', linewidth=0.5, alpha=0.8)
        current_time_sec += ep_dur
        if ep_name == 'P_US':
            ax.axvline(current_time_sec, color='white', linestyle='--', linewidth=0.5, alpha=0.8)

    # Draw horizontal lines to separate the 4 subgroups
    for boundary in subgroup_boundaries[1:-1]:
        ax.axhline(boundary, color='black', linestyle='-', linewidth=0.5, alpha=0.8)

    # Formatting Axes
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel(f'CS cells', labelpad=1)
    
    # Set requested intervals: X every 20s, Y every 100 cells
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(100))
    
    ax.tick_params(axis='both',length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Tiny Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, anchor=(1.0, 0.05))
    cbar.set_label('Z-Score', labelpad=1)
    cbar.ax.tick_params(labelsize=4.5, length=1, pad=1)
    cbar.outline.set_linewidth(0.5)

    # 6. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=600, transparent=False)
    plt.close()

## 4.2 Area-Proportional Euler diagrams for CS cells

In [ ]:
group_name = ['02.CA1-C']
group_keys = ['CA1-C'] 
group_size = len(group_name)

resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']

test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
width_mm = 30  # 
height_mm = 30 #   

for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    
    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)     # shape (n_cells, n_trials)

    trial_idx = 0
    plot_engram_euler_diagram(dict_group_response, trial_idx, width_mm, height_mm, dpath_plot, f'04_2_Area-Proportional Euler diagrams-trial-1_{group_keys[i]}')

print('All finished************')      

In [ ]:
from matplotlib_venn import venn3
def plot_engram_euler_diagram(dict_group_response, target_trial=0, width_mm=30, height_mm=30, output_path="./", title="Euler_Trial_1"):
    """
    Calculates exact ensemble intersections across all animals for a specific trial
    and plots an Area-Proportional Euler Diagram (Venn).
    target_trial: 0-indexed (0 = Trial 1).
    CS Base: #4091cf (Blue)
    US Base: #e1703c (Orange)
    pUS Base: #8cba54 (Green)
    co_cs_us: #a08085
    co_cs_pus: #66a591
    """
    set_pub_style()
    animal_keys = list(dict_group_response.keys())
    
    # Initialize subset counters
    # The 7 subsets for 3 circles: '100', '010', '110', '001', '101', '011', '111'
    # Format: CS, US, pUS
    subsets = {'100': 0, '010': 0, '001': 0, 
               '110': 0, '101': 0, '011': 0, 
               '111': 0}
               
    total_eligible_cells = 0
    total_cs = 0
    total_us = 0
    total_pus1 = 0
    total_pus2 = 0
    total_pus = 0

    # Aggregate cells across all animals for the specified trial
    for animal in animal_keys:
        flags = dict_group_response[animal]
        
        cs_t = np.array(flags['cs']).astype(bool).T[target_trial]
        us_t = np.array(flags['us']).astype(bool).T[target_trial]
        pus1_t = np.array(flags['pus_1']).astype(bool).T[target_trial]
        pus2_t = np.array(flags['pus_2']).astype(bool).T[target_trial]
        
        pus_t = pus1_t | pus2_t
        total_eligible_cells += len(cs_t)
        
        # Track total counts for metadata
        total_cs += np.sum(cs_t)
        total_us += np.sum(us_t)
        total_pus1 += np.sum(pus1_t)
        total_pus2 += np.sum(pus2_t)
        total_pus += np.sum(pus_t)
        
        # Calculate exactly which subset each cell belongs to
        # True=1, False=0. A cell that is only CS is '100'. A cell in all three is '111'.
        subsets['100'] += np.sum(cs_t & ~us_t & ~pus_t)
        subsets['010'] += np.sum(~cs_t & us_t & ~pus_t)
        subsets['001'] += np.sum(~cs_t & ~us_t & pus_t)
        
        subsets['110'] += np.sum(cs_t & us_t & ~pus_t)
        subsets['101'] += np.sum(cs_t & ~us_t & pus_t)
        subsets['011'] += np.sum(~cs_t & us_t & pus_t)
        
        subsets['111'] += np.sum(cs_t & us_t & pus_t)

    # Compile Metadata
    metadata = {
        "Figure_Title": title,
        "Trial_Index": target_trial,
        "Total_Eligible_Cells": int(total_eligible_cells),
        "Total_CS_Cells": int(total_cs),
        "Total_US_Cells": int(total_us),
        "Total_pUS1_Cells": int(total_pus1),
        "Total_pUS2_Cells": int(total_pus2),
        "Total_pUS_Union_Cells": int(total_pus),
        "Subsets_Intersection_Counts": {
            "CS_Only": int(subsets['100']),
            "US_Only": int(subsets['010']),
            "pUS_Only": int(subsets['001']),
            "CS_and_US": int(subsets['110']),
            "CS_and_pUS": int(subsets['101']),
            "US_and_pUS": int(subsets['011']),
            "CS_and_US_and_pUS": int(subsets['111'])
        }
    }

    # --- PLOTTING ---
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Professional Colors: CS (Blue), US (Orange), pUS (Green)
    v = venn3(subsets=subsets, set_labels=('CS', 'US', 'pUS'), ax=ax, 
              set_colors=('#4091cf', '#e1703c', '#8cba54'), alpha=0.6)

    # Formatting: Convert integer counts to Percentages of Total Cells
    for subset_id in subsets.keys():
        patch = v.get_patch_by_id(subset_id)
        label = v.get_label_by_id(subset_id)
        if label:
            count = subsets[subset_id]
            # Avoid division by zero
            percentage = (count / total_eligible_cells) * 100 if total_eligible_cells > 0 else 0
            
            # If percentage is too small, hide the text to prevent clutter
            if percentage > 0.5:
                label.set_text(f"{percentage:.2f}%")
                label.set_fontsize(5)
            else:
                label.set_text("")
                
            # Optional: Add subtle borders to the circles for a crisp look
            if patch:
                patch.set_edgecolor('white')
                patch.set_linewidth(0.5)

    # Format circle labels (CS, US, pUS)
    if v.set_labels:
        for text in v.set_labels:
            if text:
                text.set_fontsize(6)
                text.set_fontweight('bold')

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title)
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=300, transparent=True)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.3.1 The US stimulus triggered the reactivation of CS cells

In [ ]:
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']
test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'

width_mm = 40  # 
height_mm = 30 #   

ls_data = []
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    
    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)    
    ls_data.append(dict_group_response)
    colors = ['#e1703c', '#1f77b4']

plot_pre_us_vs_us_trigger_2groups(ls_data[0], ls_data[1], 'Co-resp / CS cells (%)', width_mm, height_mm, colors_beh_i, dpath_plot, 
                                  f'04_3_1_US triggering proof-pre_us vs us', min_base_cells=10)
print('All finished************')      

In [ ]:
def extract_compressed_metrics(dict_group, min_base_cells):
    """
    Helper function to extract denoised Trial 1 and Mean All-Trials data.
    """
    animal_keys = list(dict_group.keys())
    first_animal = animal_keys[0]
    n_trials = np.array(dict_group[first_animal]['cs']).shape[1]
    
    data_metric1 = np.zeros((n_trials, len(animal_keys))) # Pre-US
    data_metric2 = np.zeros((n_trials, len(animal_keys))) # US
    
    for a_idx, animal in enumerate(animal_keys):
        flags = dict_group[animal]
        
        cs_flags = np.array(flags['cs']).astype(bool).T      
        us_flags = np.array(flags['us']).astype(bool).T
        pre_us_flags = np.array(flags['pre_us']).astype(bool).T
        
        for t in range(n_trials):
            cs_t = cs_flags[t]
            us_t = us_flags[t]
            pre_us_t = pre_us_flags[t]
            
            n_cs = np.sum(cs_t)
            
            # DENOISING FILTER
            if n_cs < min_base_cells:
                data_metric1[t, a_idx] = np.nan
                data_metric2[t, a_idx] = np.nan
                continue
            
            data_metric1[t, a_idx] = (np.sum(cs_t & pre_us_t) / n_cs) * 100
            data_metric2[t, a_idx] = (np.sum(cs_t & us_t) / n_cs) * 100

    # Extract Trial 1
    t1_pre = data_metric1[0, :]
    t1_us = data_metric2[0, :]
    
    # Extract All Trials (Mean across trials for each animal)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        all_pre = np.nanmean(data_metric1, axis=0)
        all_us = np.nanmean(data_metric2, axis=0)
        
    return t1_pre, t1_us, all_pre, all_us

def plot_pre_us_vs_us_trigger_2groups(dict_group_c, dict_group_i, y_label, width_mm, height_mm, colors, output_path, title, min_base_cells=10):
    """
    Plots the step-function triggering effect for CA1-C and CA1-I side-by-side.
    Compresses longitudinal data into 'Trial 1' and 'All Trials' (Mean).
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    # --- 1. DATA EXTRACTION ---
    c_t1_pre, c_t1_us, c_all_pre, c_all_us = extract_compressed_metrics(dict_group_c, min_base_cells)
    i_t1_pre, i_t1_us, i_all_pre, i_all_us = extract_compressed_metrics(dict_group_i, min_base_cells)
    
    # --- 2. LAYOUT & COLOR CONFIGURATION ---
    c_dark = colors[0]       
    # Create true light solid colors to prevent stacking issues
    c_light = lighten_color(c_dark, amount=0.4) 
    
    i_dark = colors[1]       
    i_light = lighten_color(i_dark, amount=0.4)
    
    plot_configs = [
        {'name': 'CA1-C_Trial1', 'd1': c_t1_pre, 'd2': c_t1_us, 'x': 1, 'c_light': c_light, 'c_dark': c_dark},
        {'name': 'CA1-C_All',    'd1': c_all_pre, 'd2': c_all_us, 'x': 2, 'c_light': c_light, 'c_dark': c_dark},
        {'name': 'CA1-I_Trial1', 'd1': i_t1_pre, 'd2': i_t1_us, 'x': 3, 'c_light': i_light, 'c_dark': i_dark},
        {'name': 'CA1-I_All',    'd1': i_all_pre, 'd2': i_all_us, 'x': 4, 'c_light': i_light, 'c_dark': i_dark},
    ]

    # --- 3. GLOBAL Y-AXIS SCALING ---
    all_vals = []
    for cfg in plot_configs:
        all_vals.extend(cfg['d1'][~np.isnan(cfg['d1'])])
        all_vals.extend(cfg['d2'][~np.isnan(cfg['d2'])])
        
    global_max = max(all_vals) if len(all_vals) > 0 else 100
    global_min = min(all_vals) if len(all_vals) > 0 else 0
    y_range = global_max - global_min
    
    ax.set_ylim(max(0, global_min - (y_range * 0.05)), global_max + (y_range * 0.35))

    # --- 4. PLOTTING LOOP ---
    off1, off2 = -0.15, 0.15
    
    for cfg in plot_configs:
        d1_raw, d2_raw = cfg['d1'], cfg['d2']
        
        valid_mask = ~np.isnan(d1_raw) & ~np.isnan(d2_raw)
        d1 = d1_raw[valid_mask]
        d2 = d2_raw[valid_mask]
        
        if len(d1) == 0: continue
            
        x1 = cfg['x'] + off1
        x2 = cfg['x'] + off2
        
        # A. Draw Paired Animal Lines (zorder=0 puts them in the far back)
        for i in range(len(d1)):
            ax.plot([x1, x2], [d1[i], d2[i]], color='gray', alpha=0.3, lw=0.5, zorder=0)
            
        # B. Draw Scatter Dots (zorder=2)
        ax.scatter(np.full(len(d1), x1), d1, s=2.5, color=cfg['c_light'], alpha=0.9, edgecolors='none', zorder=2)
        ax.scatter(np.full(len(d2), x2), d2, s=2.5, color=cfg['c_dark'], alpha=0.9, edgecolors='none', zorder=2)
        
        # C. Draw Means and Error Bars
        m1, s1 = np.mean(d1), stats.sem(d1)
        m2, s2 = np.mean(d2), stats.sem(d2)
        
        # Mean connection line (zorder=1 so it sits behind the mean dots)
        ax.plot([x1, x2], [m1, m2], color='black', lw=1.0, zorder=1)
        
        # Error bars (zorder=4 so they sit on top of everything, blocking the black line)
        ax.errorbar(x1, m1, yerr=s1, fmt='o', color=cfg['c_light'], elinewidth=0.75, capsize=0, 
                    markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
        ax.errorbar(x2, m2, yerr=s2, fmt='o', color=cfg['c_dark'], elinewidth=0.75, capsize=0, 
                    markersize=4, markeredgecolor='white', markeredgewidth=0.25, zorder=4)
        
        # D. Paired Significance Testing (Wilcoxon)
        metadata["Statistics"][cfg['name']] = {"N_animals": len(d1),
                                              'Mean_pre-US': m1,
                                              'Mean_US': m2}
        
        if len(d1) > 3 and not np.all(d1 == d2):
            _, p_val = stats.wilcoxon(d1, d2, alternative='two-sided')
            metadata["Statistics"][cfg['name']]["PreUS_vs_US_pval"] = float(p_val)      
            star = get_asterisks(p_val)
            if star and star != 'ns':
                local_max = max(np.max(d1), np.max(d2))
                bracket_roof = local_max + (y_range * 0.05)
                
                add_stat_annotation_two_sided(ax, d1, d2, x1, x2, bracket_roof, ttest=0, paired=1)

    # --- 5. FORMATTING & AESTHETICS ---
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xticklabels(['Trial 1', 'All trials', 'Trial 1', 'All trials'], fontsize=6)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    ax.axvline(2.5, color='gray', linestyle=':', lw=0.5, alpha=0.5)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='y', length=2, pad=1)
    ax.tick_params(axis='x', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Custom Top-Tier Legend
    l_light_c = mlines.Line2D([], [], color=c_light, marker='o', linestyle='None', markersize=3)
    l_light_i = mlines.Line2D([], [], color=i_light, marker='o', linestyle='None', markersize=3)
    
    l_dark_c = mlines.Line2D([], [], color=c_dark, marker='o', linestyle='None', markersize=3)
    l_dark_i = mlines.Line2D([], [], color=i_dark, marker='o', linestyle='None', markersize=3)
    
    ax.legend(handles=[(l_light_c, l_light_i), (l_dark_c, l_dark_i)], 
              labels=['Pre-US', 'US'],
              handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
              frameon=False, loc='upper left', handletextpad=0.5, borderpad=0)

    # --- 6. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.3.2 Mean Z score plot-subgroup comparision (co_cs_us vs cs only) for CS resp cells animal-wise in CA1 

In [ ]:
def extract_PETH_mean_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp) #[:, bin_s:bin_e].mean(axis=0))        
        if len(data_trials) > 0:
            data_trials = np.concatenate(data_trials, axis=0)
            data_trials = np.nanmean(data_trials, axis=0)[bin_s:bin_e]
            data_group.append(data_trials)
    return np.array(data_group)
    
group_name = ['02.CA1-C']
group_keys = ['CA1-C'] #, 'CA1-E'
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 3 
post_du = 40 
bin_s = int((20-base_du)*fs) 
bin_e = int((20+20+20+3+ post_du)*fs) 

width_mm = 40  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

# Resp cells subgroup comparison
resp_names = ['intersec_tone_shock', 'tone_us_only', ]
resp_keys = ['CS & US', 'CS only']

for i in range(group_size): 
    dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
        
    ds_groups = {}    
    for idx, resp_name in enumerate(resp_names):
        ds_groups[resp_keys[idx]] = extract_PETH_mean_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=6)       
    
    colors = ['#a08085', '#4091cf']  
    plot_trial_population_activity(ds_groups, 'Mean Z-Score', width_mm, height_mm, epochs, resp_keys, colors, fs, dpath_plot, 
                                   f'04_3_2_Mean Z score-across all trials_{group_keys[i]}-animal-wise_{resp_keys[0]} vs {resp_keys[1]}')

print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    cal_trial : dict()--group name: (n_anmials, n_bins)
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=grp, zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.6))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    # length=2 to ensure the physical tick marks stay tiny.
    ax.legend(frameon=False, loc='upper right', handlelength=1)

    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.4 Sufficience for memory association (animal-wise) in CA1--co_resp_cell proportion accumulated

In [ ]:
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']
test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'


height_mm = 30 #   

for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    
    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)    

    # co_cs_us: #a08085
    #co_cs_pus: #66a591
    colors = ['#a08085', '#66a591']
    if i==0:
        width_mm = 35  # 
        plot_engram_accumulation_longitudinal(dict_group_response, 'Cum. cell fraction (%)', width_mm, height_mm, colors, dpath_plot, f'04_4_co_resp_cell proportion accumulated-{group_keys[i]}')
    else:
        width_mm = 40  # 
        plot_engram_accumulation_longitudinal(dict_group_response, 'Cum. cell fraction (%)', width_mm, height_mm, colors, dpath_plot, f'sup_04_4_co_resp_cell proportion accumulated-{group_keys[i]}')
print('All finished************')      

In [ ]:
def plot_engram_accumulation_longitudinal(dict_group_response, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots cumulative trial-by-trial Engram Assembly (proportion of total eligible cells).
    Shows the accumulation of CS-US (Triggering) and CS-pUS (Binding) co-firing cells.
    No significance testing; designed to show longitudinal growth.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    animal_keys = list(dict_group_response.keys())
    first_animal = animal_keys[0]
    n_trials = np.array(dict_group_response[first_animal]['cs']).shape[1]
    trials = np.arange(1, n_trials + 1)
    
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    # --- 1. CUMULATIVE DATA EXTRACTION ---
    data_metric1 = np.zeros((n_trials, len(animal_keys))) # Cumulative CS & US
    data_metric2 = np.zeros((n_trials, len(animal_keys))) # Cumulative CS & pUS
    
    for a_idx, animal in enumerate(animal_keys):
        flags = dict_group_response[animal]
        
        cs_flags = np.array(flags['cs']).astype(bool).T      
        us_flags = np.array(flags['us']).astype(bool).T
        pus1_flags = np.array(flags['pus_1']).astype(bool).T
        pus2_flags = np.array(flags['pus_2']).astype(bool).T
        
        pus_flags = pus1_flags | pus2_flags
        total_cells = cs_flags.shape[1]
        
        for t in range(n_trials):
            if total_cells == 0:
                data_metric1[t, a_idx] = np.nan
                data_metric2[t, a_idx] = np.nan
                continue
            
            # Create a running union up to the current trial 't'
            cum_cs_us = np.zeros(total_cells, dtype=bool)
            cum_cs_pus = np.zeros(total_cells, dtype=bool)
            
            for curr_t in range(t + 1):
                cs_curr = cs_flags[curr_t]
                us_curr = us_flags[curr_t]
                pus_curr = pus_flags[curr_t]
                
                # Logical OR with previous trials to accumulate cells
                cum_cs_us |= (cs_curr & us_curr)
                cum_cs_pus |= (cs_curr & pus_curr)
            
            # Calculate Absolute Proportion of all cells
            prop_cs_us = np.sum(cum_cs_us) / total_cells * 100
            prop_cs_pus = np.sum(cum_cs_pus) / total_cells * 100
            
            data_metric1[t, a_idx] = prop_cs_us
            data_metric2[t, a_idx] = prop_cs_pus

    # --- 2. GLOBAL Y-AXIS SCALING ---
    all_vals_1 = [v for trial in data_metric1 for v in trial if not np.isnan(v)]
    all_vals_2 = [v for trial in data_metric2 for v in trial if not np.isnan(v)]
    all_vals = all_vals_1 + all_vals_2
    
    global_max = max(all_vals) if len(all_vals) > 0 else 100
    global_min = 0 # Force minimum to 0 for cumulative plots
    y_range = global_max - global_min
    
    ax.set_ylim(0, global_max + (y_range * 0.15))

    # --- 3. PLOTTING LOOP ---
    off_1 = -0.15 
    off_2 = 0.15  
    means_1, means_2 = [], []
    sems_1, sems_2 = [], []
    
    for t_idx in range(n_trials):
        d1, d2 = data_metric1[t_idx], data_metric2[t_idx]
        d1, d2 = d1[~np.isnan(d1)], d2[~np.isnan(d2)]
        
        m1, s1 = np.mean(d1), stats.sem(d1)
        m2, s2 = np.mean(d2), stats.sem(d2)
        means_1.append(m1); sems_1.append(s1)
        means_2.append(m2); sems_2.append(s2)
        
        jitter = 0.08
        x1_scatter = (trials[t_idx] + off_1) + np.random.uniform(-jitter, jitter, size=len(d1))
        x2_scatter = (trials[t_idx] + off_2) + np.random.uniform(-jitter, jitter, size=len(d2))
        
        ax.scatter(x1_scatter, d1, s=2.5, color=colors[0], alpha=0.3, edgecolors='none', zorder=1)
        ax.scatter(x2_scatter, d2, s=2.5, color=colors[1], alpha=0.3, edgecolors='none', zorder=1)

    # --- 4. CONNECT MEANS & ERROR BARS ---
    ax.plot(trials + off_1, means_1, color=colors[0], lw=0.75, alpha=0.8, zorder=2, label="CS & US")
    ax.plot(trials + off_2, means_2, color=colors[1], lw=0.75, alpha=0.8, zorder=2, label="CS & pUS")
    
    ax.errorbar(trials + off_1, means_1, yerr=sems_1, fmt='o', 
                color=colors[0], elinewidth=0.75, capsize=0, markersize=3.5, 
                markeredgecolor='white', markeredgewidth=0.25, zorder=3)
                
    ax.errorbar(trials + off_2, means_2, yerr=sems_2, fmt='o', 
                color=colors[1], elinewidth=0.75, capsize=0, markersize=3.5, 
                markeredgecolor='white', markeredgewidth=0.25, zorder=3)

    # --- 5. FORMATTING & AESTHETICS ---
    ax.set_xticks(trials)
    ax.set_xlabel('Trial number', labelpad=1)
    ax.set_ylabel(y_label,  labelpad=0.1)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    ax.legend(frameon=False, loc='upper left', handlelength=1.5)

    # --- 6. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.5.1 Cross-registration to recall session_curve

In [ ]:
test_algori_data = 'post_03_5_cross_registration_to_recall' 
dpath_data = os.path.join(dpath_cal_all, test_algori_data)

epochs = {'Base': 20, 'CS_re': 30, 'pCS_re': 20}
height_mm = 30 #  

with open(os.path.join(dpath_data, f'02.CA1-C_ds_cal_recall_animal_wise.pkl'), 'rb') as f:
    ds_animal_wise = pickle.load(f) 

# co_cs_both (union of us and pus) plotting
colors = ['#66a591', '#4091cf', '#7f7f7f']
engram_key = 'co_cs_both'
width_mm = 40  # 
plot_CR_to_recall_dynamics_animal_wise(ds_animal_wise, engram_key, epochs, width_mm, height_mm, colors, dpath_plot, 
                                        f'04_5_1_Mean z score of cross registered cells_co_cs_both in recall session_CA1-C')  
print('All finished************') 

In [ ]:
def plot_CR_to_recall_dynamics_animal_wise(ds_animal_wise, engram_key, epochs, width_mm, height_mm, colors, dpath_plot, title, flag_smooth=1, min_cells=4, fs=5):
    """
    Plots the animal-wise mean Z-scores for recall dynamics, averaged across all 4 trials.
    Drops an animal entirely if any of its target ensembles have fewer than min_cells.
    """
    set_pub_style()
    
    engram_list = ds_animal_wise.get(f'{engram_key}_engram', [])
    non_engram_list = ds_animal_wise.get(f'{engram_key}_non_engram', [])
    random_list = ds_animal_wise.get(f'{engram_key}_random', [])
    
    n_animals = len(engram_list)
    
    engram_means = []
    non_engram_means = []
    random_means = []
    
    # --- 1. DATA EXTRACTION & DENOISING ---
    for i in range(n_animals):
        e_arr = engram_list[i]
        ne_arr = non_engram_list[i]
        r_arr = random_list[i]
        
        # Check if arrays are valid and have the correct 3D shape (trials, cells, bins)
        if e_arr.size == 0 or ne_arr.size == 0 or r_arr.size == 0 or e_arr.ndim != 3:
            continue
            
        # Enforce min_cells strict pairing (Axis 1 is the cell dimension)
        if e_arr.shape[1] >= min_cells and ne_arr.shape[1] >= min_cells and r_arr.shape[1] >= min_cells:
            # Average across cells for this specific animal -> shape (n_trials, n_bins)
            engram_means.append(np.nanmean(e_arr, axis=1))
            non_engram_means.append(np.nanmean(ne_arr, axis=1))
            random_means.append(np.nanmean(r_arr, axis=1))
            
    # Stack back into (n_valid_animals, n_trials, n_bins) arrays
    engram_aw = np.stack(engram_means) if engram_means else np.array([])
    non_engram_aw = np.stack(non_engram_means) if non_engram_means else np.array([])
    random_aw = np.stack(random_means) if random_means else np.array([])
    
    cal_trial = {}
    
    # Trial-Averaged data: Average across trials (axis 1) for each animal
    # Output Shape becomes (n_valid_animals, n_bins)
    if engram_aw.size > 0: cal_trial['Engram'] = np.nanmean(engram_aw, axis=1)
    if non_engram_aw.size > 0: cal_trial['Non-Engram'] = np.nanmean(non_engram_aw, axis=1)
    if random_aw.size > 0: cal_trial['Random'] = np.nanmean(random_aw, axis=1)

    # Apply Gaussian Smoothing
    if flag_smooth == 1:
        sigma_val = max(1, int(fs / 2)) # ~0.5s smoothing window
        for k in cal_trial.keys():
            cal_trial[k] = gaussian_filter1d(cal_trial[k], sigma=sigma_val, axis=-1)

    # --- 2. SETUP CANVAS & METADATA ---
    group_keys = ['Engram', 'Non-Engram', 'Random']
    group_labels = ['CS & US/pUS', 'CS only *', 'Non-CS **']
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 3. TIME VECTOR & EPOCH SHADING ---
    t_start = -epochs.get('Base', 5.0) 
    
    if not cal_trial:
        print(f"Skipping {title}: No valid animals passed the min_cells={min_cells} threshold.")
        return
        
    first_key = list(cal_trial.keys())[0]
    n_bins = cal_trial[first_key].shape[1]
    x_time = np.arange(n_bins) / fs + t_start

    # Flexible shading map for both classical and recall naming conventions
    epoch_shading_map = {
        'CS_re': ('#4091cf', 0.1),    
        'pCS_re': ('gray', 0.05),     
    }        
    
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue 
            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 4. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_valid_animals = data.shape[0]
        
        if n_valid_animals == 0: continue
        
        mean_curve = np.nanmean(data, axis=0)
        # Calculate SEM across the valid animals
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_valid_animals)
        color = colors[i]        
        
        # Plot mean line
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=group_labels[i], zorder=3)
        
        # Weaker shading (alpha=0.15) for cleaner overlaps
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.15, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {
            "N_animals_survived": n_valid_animals,
            "Peak_Z_Score": float(np.nanmax(mean_curve))
        }

    # --- 5. PUBLICATION FORMATTING (Low Ink Standard) ---
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel('Mean Z-Score', labelpad=0.1)
    
    ax.set_xlim(t_start, x_time[-1])
    
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    
    ax.tick_params(axis='both', length=2, pad=1) 
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    
    # Minimalist legend
    ax.legend(loc='best', frameon=False, handlelength=1.5, handletextpad=0.4, fontsize=6)

    # --- 6. EXPORT ---
    os.makedirs(dpath_plot, exist_ok=True)
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, dpath_plot, title)

## supp_4.5.1 Cross-registration to recall session_stat

In [ ]:
test_algori_data = 'post_03_5_cross_registration_to_recall' 
dpath_data = os.path.join(dpath_cal_all, test_algori_data)

epochs = {'Base': 20, 'CS_re': 30, 'pCS_re': 20}
height_mm = 30 #  

with open(os.path.join(dpath_data, f'02.CA1-C_ds_cal_recall_animal_wise.pkl'), 'rb') as f:
    ds_animal_wise = pickle.load(f) 

# co_cs_both (union of us and pus) plotting
colors = ['#66a591', '#4091cf', '#7f7f7f']
engram_key = 'co_cs_both'
width_mm = 35  # 
plot_CR_to_recall_stat_animal_wise(ds_animal_wise, engram_key, epochs, width_mm, height_mm, colors, dpath_plot, 
                                  f'sup_04_5_1_Stat of cross registered cells_co_cs_both in recall session_CA1-C')   
print('All finished************') 

In [ ]:
def plot_CR_to_recall_stat_animal_wise(ds_animal_wise, engram_key, epochs, width_mm, height_mm, colors, dpath_plot, title, min_cells=4, fs=5):
    """
    Quantifies and plots the animal-wise peak Z-scores during the first 3 seconds of CS recall.
    Performs Wilcoxon paired tests between overlapping vs CS only, and overlapping vs non-CS.
    """
    set_pub_style()
    
    engram_list = ds_animal_wise.get(f'{engram_key}_engram', [])
    non_engram_list = ds_animal_wise.get(f'{engram_key}_non_engram', [])
    random_list = ds_animal_wise.get(f'{engram_key}_random', [])
    
    n_animals = len(engram_list)
    
    peaks_engram = []
    peaks_non_engram = []
    peaks_random = []
    
    # Window for peak extraction: First 3s of CS
    base_dur = epochs.get('Base', 20.0)
    cs_start_bin = int(base_dur * fs)
    cs_end_bin = int((base_dur + 3.0) * fs)
    
    # --- 1. DATA EXTRACTION & PEAK CALCULATION ---
    for i in range(n_animals):
        e_arr = engram_list[i]
        ne_arr = non_engram_list[i]
        r_arr = random_list[i]
        
        # Check validity and dimensions
        if e_arr.size == 0 or ne_arr.size == 0 or r_arr.size == 0 or e_arr.ndim != 3:
            continue
            
        # Enforce min_cells strict pairing (Axis 1 is the cell dimension)
        if e_arr.shape[1] >= min_cells and ne_arr.shape[1] >= min_cells and r_arr.shape[1] >= min_cells:
            # Step 1: Average across cells -> shape (trials, bins)
            e_cell_mean = np.nanmean(e_arr, axis=1)
            ne_cell_mean = np.nanmean(ne_arr, axis=1)
            r_cell_mean = np.nanmean(r_arr, axis=1)
            
            # Step 2: Average across trials -> shape (bins,)
            e_trial_mean = np.nanmean(e_cell_mean, axis=0)
            ne_trial_mean = np.nanmean(ne_cell_mean, axis=0)
            r_trial_mean = np.nanmean(r_cell_mean, axis=0)
            
            # Step 3: Extract peak in the 3s window
            peaks_engram.append(np.nanmax(e_trial_mean[cs_start_bin:cs_end_bin]))
            peaks_non_engram.append(np.nanmax(ne_trial_mean[cs_start_bin:cs_end_bin]))
            peaks_random.append(np.nanmax(r_trial_mean[cs_start_bin:cs_end_bin]))

    # Convert to arrays for plotting
    d_e = np.array(peaks_engram)
    d_ne = np.array(peaks_non_engram)
    d_r = np.array(peaks_random)

    if len(d_e) == 0:
        print(f"Skipping {title}: No valid animals passed criteria.")
        return

    # --- 2. SETUP CANVAS & METADATA ---
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Statistics_Wilcoxon": {"N_Animals": int(len(d_e))}}
    
    # --- 3. GLOBAL Y-AXIS SCALING ---
    all_vals = np.concatenate([d_e, d_ne, d_r])
    global_max = max(all_vals) if len(all_vals) > 0 else 5
    global_min = min(all_vals) if len(all_vals) > 0 else 0
    data_range = global_max - global_min if (global_max - global_min) > 0 else 1.0
    
    # NOTE: Setting Y-limits here is crucial so `ax.get_ylim()` works perfectly for the significance bars later
    ax.set_ylim(min(0, global_min - (data_range * 0.05)), global_max + (data_range * 0.45))

    # --- 4. PLOTTING PAIRED DATA ---
    x_e, x_ne, x_r = 1, 2, 3
    
    # A. Draw Paired Animal Lines (zorder=0)
    for i in range(len(d_e)):
        # Connect Engram to Non-Engram, and Non-Engram to Random
        ax.plot([x_e, x_ne], [d_e[i], d_ne[i]], color='gray', alpha=0.3, lw=0.5, zorder=0)
        ax.plot([x_ne, x_r], [d_ne[i], d_r[i]], color='gray', alpha=0.3, lw=0.5, zorder=0)
        
    # B. Draw Scatter Dots (zorder=2)
    ax.scatter(np.full(len(d_e), x_e), d_e, s=2.5, color=colors[0], alpha=0.9, edgecolors='none', zorder=2)
    ax.scatter(np.full(len(d_ne), x_ne), d_ne, s=2.5, color=colors[1], alpha=0.9, edgecolors='none', zorder=2)
    ax.scatter(np.full(len(d_r), x_r), d_r, s=2.5, color=colors[2], alpha=0.9, edgecolors='none', zorder=2)

    # C. Draw Means and Error Bars
    m_e, s_e = np.mean(d_e), stats.sem(d_e)
    m_ne, s_ne = np.mean(d_ne), stats.sem(d_ne)
    m_r, s_r = np.mean(d_r), stats.sem(d_r)
    
    # Mean connection lines
    ax.plot([x_e, x_ne], [m_e, m_ne], color='black', lw=1.0, zorder=1)
    ax.plot([x_ne, x_r], [m_ne, m_r], color='black', lw=1.0, zorder=1)
    
    # Error bars block the black line
    ax.errorbar(x_e, m_e, yerr=s_e, fmt='o', color=colors[0], elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
    ax.errorbar(x_ne, m_ne, yerr=s_ne, fmt='o', color=colors[1], elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
    ax.errorbar(x_r, m_r, yerr=s_r, fmt='o', color=colors[2], elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)

    # --- 5. STATISTICAL TESTING & BRACKETS ---
    current_roof = global_max + (data_range * 0.05)
    
    if len(d_e) > 3:
        # Engram vs Non-Engram
        if not np.all(d_e == d_ne):
            _, p_e_ne = stats.wilcoxon(d_e, d_ne, alternative='two-sided')
            metadata["Statistics_Wilcoxon"]["Engram_vs_NonEngram_pval"] = float(p_e_ne)
            
            # Using your imported get_asterisks function
            try: star_e_ne = get_asterisks(p_e_ne)
            except NameError: 
                star_e_ne = '***' if p_e_ne < 0.001 else '**' if p_e_ne < 0.01 else '*' if p_e_ne < 0.05 else 'ns'
                
            if star_e_ne != 'ns':
                current_roof = add_significance_bar(ax, x_e, x_ne, current_roof, star_e_ne)
                
        # Engram vs Random (Stacking above the previous bracket if necessary)
        if not np.all(d_e == d_r):
            _, p_e_r = stats.wilcoxon(d_e, d_r, alternative='two-sided')
            metadata["Statistics_Wilcoxon"]["Engram_vs_Random_pval"] = float(p_e_r)
            
            try: star_e_r = get_asterisks(p_e_r)
            except NameError: 
                star_e_r = '***' if p_e_r < 0.001 else '**' if p_e_r < 0.01 else '*' if p_e_r < 0.05 else 'ns'
                
            if star_e_r != 'ns':
                _ = add_significance_bar(ax, x_e, x_r, current_roof, star_e_r)

    # --- 6. FORMATTING & AESTHETICS ---
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['CS &\nUS/pUS', 'CS\nonly', 'Non-\nCS'], fontsize=7)
    ax.set_ylabel('Peak Z-Score', labelpad=0.1)
    
    ax.set_xlim(0.5, 3.5)
    
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.4))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 7. EXPORT ---
    os.makedirs(dpath_plot, exist_ok=True)
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()  
    save_metadata_json(metadata, dpath_plot, title)

## 4.5.2 Change from condi to recall

In [ ]:
test_algori_data = 'post_03_5_cross_registration_to_recall' 
dpath_data = os.path.join(dpath_cal_all, test_algori_data)

epochs_condi = {'Base': 20, 'CS': 20, 'pCS': 20}
epochs_recall = {'Base': 20, 'CS_re': 30, 'pCS_re': 20}
height_mm = 30 #  

with open(os.path.join(dpath_data, f'02.CA1-C_ds_cal_condi_animal_wise.pkl'), 'rb') as f:
    ds_animal_wise_condi = pickle.load(f)
     
with open(os.path.join(dpath_data, f'02.CA1-C_ds_cal_recall_animal_wise.pkl'), 'rb') as f:
    ds_animal_wise_recall = pickle.load(f) 

# co_cs_both (union of us and pus) plotting
colors = ['#66a591', '#4091cf', '#7f7f7f']
engram_key = 'co_cs_both'
width_mm = 35  # 
plot_plasticity_stat_animal_wise(ds_animal_wise_condi, ds_animal_wise_recall, engram_key, epochs_condi, epochs_recall, width_mm, height_mm, colors, dpath_plot, 
                                f'04_5_2_Stat of plasticity_co_cs_both from condi to recall session_CA1-C')

print('All finished************') 

In [ ]:
def get_n_cells(arr):
    return arr.shape[1] if arr.ndim == 3 else arr.shape[0] if arr.ndim == 2 else 0

def get_peak(arr, start_bin, end_bin):
    if arr.ndim == 3: # (trials, cells, bins)
        return np.nanmax(np.nanmean(np.nanmean(arr, axis=1), axis=0)[start_bin:end_bin])
    elif arr.ndim == 2: # (cells, bins)
        return np.nanmax(np.nanmean(arr, axis=0)[start_bin:end_bin])
    return np.nan
    
def calc_mi(p_r, p_c):
    """
    Calculates Z score change
    """
    return (p_r - p_c)

def plot_plasticity_stat_animal_wise(ds_animal_wise_condi, ds_animal_wise_recall, engram_key, epochs_condi, epochs_recall, width_mm, height_mm, colors, dpath_plot, title, min_cells=4, fs=5):
    """
    Calculates the Plasticity induced response change: (Recall - Condi) 
    Robustly handles negative Z-scores representing sparse/suppressed states.
    Plots paired statistics
    """
    set_pub_style()
    
    # 1. EXTRACT LISTS
    e_list_c = ds_animal_wise_condi.get(f'{engram_key}_engram', [])
    ne_list_c = ds_animal_wise_condi.get(f'{engram_key}_non_engram', [])
    r_list_c = ds_animal_wise_condi.get(f'{engram_key}_random', [])
    
    e_list_r = ds_animal_wise_recall.get(f'{engram_key}_engram', [])
    ne_list_r = ds_animal_wise_recall.get(f'{engram_key}_non_engram', [])
    r_list_r = ds_animal_wise_recall.get(f'{engram_key}_random', [])
    
    n_animals = min(len(e_list_c), len(e_list_r))
    
    mi_engram, mi_non_engram, mi_random = [], [], []
    
    # Define timing bounds
    c_base = epochs_condi.get('Base', 20.0)
    c_start = int(c_base * fs)
    c_end = int((c_base + 3.0) * fs)
    
    r_base = epochs_recall.get('Base', 20.0)
    r_start = int(r_base * fs)
    r_end = int((r_base + 3.0) * fs)

    # 2. CALCULATE MODULATION INDEX
    for i in range(n_animals):
        e_c, ne_c, r_c = e_list_c[i], ne_list_c[i], r_list_c[i]
        e_r, ne_r, r_r = e_list_r[i], ne_list_r[i], r_list_r[i]
        
        # Check validity (Not empty and valid dimensions)
        if e_c.size == 0 or ne_c.size == 0 or r_c.size == 0 or e_c.ndim < 2: continue
        if e_r.size == 0 or ne_r.size == 0 or r_r.size == 0 or e_r.ndim < 2: continue

        
        if get_n_cells(e_c) >= min_cells and get_n_cells(ne_c) >= min_cells and get_n_cells(r_c) >= min_cells and \
           get_n_cells(e_r) >= min_cells and get_n_cells(ne_r) >= min_cells and get_n_cells(r_r) >= min_cells:
            
            # Condi Peaks
            peak_e_c = get_peak(e_c, c_start, c_end)
            peak_ne_c = get_peak(ne_c, c_start, c_end)
            peak_r_c = get_peak(r_c, c_start, c_end)

            # Recall Peaks
            peak_e_r = get_peak(e_r, r_start, r_end)
            peak_ne_r = get_peak(ne_r, r_start, r_end)
            peak_r_r = get_peak(r_r, r_start, r_end)

            # Calculate delta
            mi_engram.append(calc_mi(peak_e_r, peak_e_c))
            mi_non_engram.append(calc_mi(peak_ne_r, peak_ne_c))
            mi_random.append(calc_mi(peak_r_r, peak_r_c))

    d_e = np.array(mi_engram)
    d_ne = np.array(mi_non_engram)
    d_r = np.array(mi_random)

    if len(d_e) == 0:
        print(f"Skipping {title}: No valid animals passed criteria.")
        return

    # 3. SETUP CANVAS & METADATA
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Statistics_Wilcoxon": {"N_Animals": int(len(d_e))}}
    
    # 4. GLOBAL Y-AXIS SCALING
    all_vals = np.concatenate([d_e, d_ne, d_r])
    global_max = max(all_vals) if len(all_vals) > 0 else 1.0
    global_min = min(all_vals) if len(all_vals) > 0 else -1.0
    data_range = global_max - global_min if (global_max - global_min) > 0 else 2.0
    
    ax.set_ylim(global_min - (data_range * 0.1), global_max + (data_range * 0.4))

    # 5. PLOTTING PAIRED DATA
    x_e, x_ne, x_r = 1, 2, 3
    
    # Add baseline indicator at MI = 0 (No plasticity)
    ax.axhline(0, color='black', linestyle='--', lw=0.5, alpha=0.5, zorder=0)
    
    # A. Draw Paired Animal Lines
    for i in range(len(d_e)):
        ax.plot([x_e, x_ne], [d_e[i], d_ne[i]], color='gray', alpha=0.3, lw=0.5, zorder=0)
        ax.plot([x_ne, x_r], [d_ne[i], d_r[i]], color='gray', alpha=0.3, lw=0.5, zorder=0)
        
    # B. Draw Scatter Dots
    ax.scatter(np.full(len(d_e), x_e), d_e, s=2.5, color=colors[0], alpha=0.9, edgecolors='none', zorder=2)
    ax.scatter(np.full(len(d_ne), x_ne), d_ne, s=2.5, color=colors[1], alpha=0.9, edgecolors='none', zorder=2)
    ax.scatter(np.full(len(d_r), x_r), d_r, s=2.5, color=colors[2], alpha=0.9, edgecolors='none', zorder=2)

    # C. Draw Means and Error Bars
    m_e, s_e = np.mean(d_e), stats.sem(d_e)
    m_ne, s_ne = np.mean(d_ne), stats.sem(d_ne)
    m_r, s_r = np.mean(d_r), stats.sem(d_r)
    
    ax.plot([x_e, x_ne], [m_e, m_ne], color='black', lw=1.0, zorder=1)
    ax.plot([x_ne, x_r], [m_ne, m_r], color='black', lw=1.0, zorder=1)
    
    ax.errorbar(x_e, m_e, yerr=s_e, fmt='o', color=colors[0], elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
    ax.errorbar(x_ne, m_ne, yerr=s_ne, fmt='o', color=colors[1], elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
    ax.errorbar(x_r, m_r, yerr=s_r, fmt='o', color=colors[2], elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)

    # 6. STATISTICAL TESTING & BRACKETS
    current_roof = global_max + (data_range * 0.05)
    
    if len(d_e) > 3:
        if not np.all(d_e == d_ne):
            _, p_e_ne = stats.wilcoxon(d_e, d_ne, alternative='two-sided')
            metadata["Statistics_Wilcoxon"]["Engram_vs_NonEngram_pval"] = float(p_e_ne)
            
            star_e_ne = get_asterisks(p_e_ne)
            if star_e_ne != 'ns':
                current_roof = add_significance_bar(ax, x_e, x_ne, current_roof, star_e_ne)
                
        if not np.all(d_e == d_r):
            _, p_e_r = stats.wilcoxon(d_e, d_r, alternative='two-sided')
            metadata["Statistics_Wilcoxon"]["Engram_vs_Random_pval"] = float(p_e_r)
            
            star_e_r = get_asterisks(p_e_r)
            if star_e_r != 'ns':
                _ = add_significance_bar(ax, x_e, x_r, current_roof, star_e_r)

    # 7. FORMATTING & AESTHETICS
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['CS &\nUS/pUS', 'CS\nonly', 'Non-\nCS'], fontsize=7)
    
    ax.set_ylabel('Δ Z Score\n(Recall - Condi)', labelpad=0.1)
    ax.set_xlim(0.5, 3.5)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # 8. EXPORT
    os.makedirs(dpath_plot, exist_ok=True)
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, dpath_plot, title)

## 4.6 Mean z score plotting of Co-resp cells (co-cs-us, co-cs-pus-1, co-cs-pus_2) and CS only cells animal-wise in CA1

In [ ]:
def extract_PETH_mean_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp) #[:, bin_s:bin_e].mean(axis=0))        
        if len(data_trials) > 0:
            data_trials = np.concatenate(data_trials, axis=0)
            data_trials = np.nanmean(data_trials, axis=0)[bin_s:bin_e]
            data_group.append(data_trials)
        else:
            data_group.append(np.nan)
    return np.array(data_group)
    
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 3 # 10s
post_du = 40 #post shock  use 20s
bin_s = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
bin_e = int((20+20+20+3+ post_du)*fs) # post-shock 17s

width_mm = 40  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

resp_names = ['intersec_tone_shock', 'intersec_tone_p_shock1', 'intersec_tone_p_shock2']#  'tone', 'trace',
resp_keys = ['co_cs_us', 'co_cs_pus_1', 'co_cs_pus_2']

for idx, resp_name in enumerate(resp_names):
    ds_groups = {}
    for i in range(group_size):    
        dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
        with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
            cal_tmp = pickle.load(f)
        ds_groups[group_keys[i]] = extract_PETH_mean_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=2)  

    # 1. plot of first 2 trials -Mean Z score 
    plot_trial_population_activity(ds_groups, 'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 
                                   f'04_6_{idx+1}_First 2 trials-Mean Z score-CA1-animal-wise_{resp_keys[idx]} resp cells')
    
print('All finished************') 

In [ ]:
# For CS only cells not responsive to US, pUS-1, pUS-2
resp_names = ['tone_us_only', 'tone_p1_only', 'tone_p2_only']
resp_keys = ['cs_only_us', 'cs_only_p1', 'cs_only_p2']

for idx, resp_name in enumerate(resp_names):
    ds_groups = {}
    for i in range(group_size):    
        dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
        with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
            cal_tmp = pickle.load(f)
        ds_groups[group_keys[i]] = extract_PETH_mean_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=2)  

    # 1. plot of ALl 6 trials -Mean Z score 
    plot_trial_population_activity(ds_groups, 'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 
                                   f'sup_04_6_{idx+1}_First 2 trials-Mean Z score-CA1-animal-wise_{resp_keys[idx]} resp cells')

print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the TFC trials.
    Locked to strict millimeter layout (e.g., 60x30mm).
    cal_trial : dict()--group name: (n_anmials, n_bins)
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',  labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.7 Co-resp cells Mean z score statictics (co-cs-us, co-cs-pus-1, co-cs-pus_2) animal-wise in CA1

In [ ]:
def extract_PETH_first_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        data_tmps = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_tmps.append(tmp[:, bin_s:bin_e]) #[:, bin_s:bin_e].mean(axis=0))        
            if trial_idx ==0:
                if len(data_tmps) >0:
                    data_trials.append(data_tmps[0])
                else:
                    data_trials.append(np.nan)
            else:   
                if len(data_tmps) > 0:
                    data_trials_extract = np.concatenate(data_tmps, axis=0)
                    data_trials.append(data_trials_extract)
                else:
                    data_trials.append(np.nan)                
        data_group.append(data_trials)
    return data_group
    
def extract_PETH_each_trial(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp[:, bin_s:bin_e]) #[:, bin_s:bin_e].mean(axis=0))
            else:
                data_trials.append(np.nan)
        data_group.append(data_trials)
    return data_group   
   
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 20 # 10s
post_du1 = 3
post_du2 = 6 #post shock  use 20s
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)), 
    'pus_2': (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

#intersec_'+g1+'_'+g2
resp_names = ['intersec_tone_shock', 'intersec_tone_p_shock1', 'intersec_tone_p_shock2']#  'tone', 'trace',
resp_keys = ['co_cs_us', 'co_cs_pus_1', 'co_cs_pus_2']
epoch_names = ['us', 'pus_1', 'pus_2']

# Merge all 3 co-resp groups
width_mm = 70  # 
height_mm = 30 #  

ds_groups = {}
for i in range(group_size):    
    
    dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
    
    ds_resp_data = {}
    for idx, resp_name in enumerate(resp_names):
        bin_s, bin_e = epochs[epoch_names[idx]]
        ds_resp_data[resp_keys[idx]] = extract_PETH_first_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=6)

    ds_groups[group_keys[i]] = ds_resp_data
    
plot_mean_z_score_stat_animal_wise(ds_groups, 'Mean Z-Score', resp_keys, width_mm, height_mm, group_keys, colors_beh_i, dpath_plot, 
                       f'04_7_Mean Z score statistics of co reps cells-CA1-animal-wise', min_base_cells=4)
print('All finished************') 

In [ ]:
def plot_mean_z_score_stat_animal_wise(ds_group, y_label, resp_keys, width_mm, height_mm, group_keys, colors, output_path, title, min_base_cells=4):
    """
    Plots categorized Mean Z-scores (Initial, Early, Late) for 3 co-response subgroups in a 1x3 grid.
    Includes denoising: Conditions with fewer than `min_base_cells` are excluded.
    
    Parameters:
    - ds_group: Dict containing animal lists for each group and subgroup.
      Structure: ds_group[group_key][resp_key] = [animal_1, animal_2, ...]
      Where animal_x = [trial_1_array, trial_2_array, ...] (arrays are n_cells x n_bins, already pooled)
    """
    set_pub_style()
    resp_titles = ['CS $\cap$ US', 'CS $\cap$ pUS$_1$', 'CS $\cap$ pUS$_2$']    
    # Conditions: [Name, Target Trial Number (1-indexed)]
    conditions = [
        ('Initial', 1), # Trial 1
        ('Early', 2),   # First 2 trials (Union pooled at index 1)
        ('Late', 6)     # All 6 trials (Union pooled at index 5)
    ]
    x_labels = ['Initial\n(T1)', 'Early\n(T1-2)', 'All\n(T1-6)']
    
    # --- 1. DATA EXTRACTION & DENOISING ---
    master_data = {r_key: {} for r_key in resp_keys}
    
    for r_key in resp_keys:
        for g_key in group_keys:
            animal_list = ds_group.get(g_key, {}).get(r_key, [])
            n_animals = len(animal_list)
            
            metric_matrix = np.full((len(conditions), n_animals), np.nan)
            
            for a_idx, animal_trials in enumerate(animal_list):
                for c_idx, (c_name, t_num) in enumerate(conditions):
                    
                    t_idx = t_num - 1 # Convert 1-indexed trial number to 0-indexed list index
                    
                    if t_idx < len(animal_trials):
                        trial_data = animal_trials[t_idx]                        
                        # Validate the array
                        if isinstance(trial_data, np.ndarray) and trial_data.ndim == 2:                          
                            # Apply Denoising Threshold
                            if trial_data.shape[0] >= min_base_cells:
                                with warnings.catch_warnings():
                                    warnings.simplefilter("ignore", category=RuntimeWarning)
                                    metric_matrix[c_idx, a_idx] = np.nanmean(trial_data)
                                
            master_data[r_key][g_key] = metric_matrix
    # --- 2. GLOBAL Y-AXIS SCALING ---
    all_vals = []
    for r_key in resp_keys:
        for g_key in group_keys:
            matrix = master_data[r_key][g_key]
            all_vals.extend(matrix[~np.isnan(matrix)])
            
    if len(all_vals) == 0:
        raise ValueError(f"No valid data found after filtering (min_base_cells={min_base_cells}).")
        
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min if global_max != global_min else 1.0
    
    shared_ymin = global_min - (y_range * 0.05)
    shared_ymax = global_max + (y_range * 0.25) # Headroom for brackets

    # --- 3. CANVAS SETUP (1x3 Facet Grid) ---
    fig, axes = plt.subplots(1, 3, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    x_positions = np.array([1, 2, 3])
    off_1, off_2 = -0.15, 0.15  

    # --- 4. PLOTTING LOOP ---
    for ax_idx, (ax, r_key, r_title) in enumerate(zip(axes, resp_keys, resp_titles)):
        metadata["Statistics"][r_key] = {}
        
        g1_matrix = master_data[r_key][group_keys[0]]
        g2_matrix = master_data[r_key][group_keys[1]]
        
        ax.set_ylim(shared_ymin, shared_ymax)
        ax.set_xlim(0.5, 3.5)
        
        for c_idx, (c_name, _) in enumerate(conditions):
            d1 = g1_matrix[c_idx, :]
            d2 = g2_matrix[c_idx, :]
            
            # Mask out NaNs
            d1 = d1[~np.isnan(d1)]
            d2 = d2[~np.isnan(d2)]
            
            # Stats
            m1 = np.mean(d1) if len(d1) > 0 else np.nan
            s1 = stats.sem(d1) if len(d1) > 1 else np.nan
            m2 = np.mean(d2) if len(d2) > 0 else np.nan
            s2 = stats.sem(d2) if len(d2) > 1 else np.nan
            
            x_base = x_positions[c_idx]
            x1 = x_base + off_1
            x2 = x_base + off_2
            
            # Scatter Dots
            jitter = 0.06
            x1_scatter = x1 + np.random.uniform(-jitter, jitter, size=len(d1))
            x2_scatter = x2 + np.random.uniform(-jitter, jitter, size=len(d2))
            
            ax.scatter(x1_scatter, d1, s=2.0, color=colors[0], alpha=0.4, edgecolors='none', zorder=1)
            ax.scatter(x2_scatter, d2, s=2.0, color=colors[1], alpha=0.4, edgecolors='none', zorder=1)
            
            # Error Bars & Mean
            if not np.isnan(m1):
                ax.errorbar(x1, m1, yerr=s1, fmt='o', color=colors[0], elinewidth=0.75, capsize=0, 
                            markersize=3.5, markeredgecolor='white', markeredgewidth=0.25, zorder=3)
            if not np.isnan(m2):
                ax.errorbar(x2, m2, yerr=s2, fmt='o', color=colors[1], elinewidth=0.75, capsize=0, 
                            markersize=3.5, markeredgecolor='white', markeredgewidth=0.25, zorder=3)
            
            # Significance Testing
            stat_key = f"{c_name}"
            metadata["Statistics"][r_key][stat_key] = {
                f"N_{group_keys[0]}": len(d1),
                f"N_{group_keys[1]}": len(d2),
                f"N_{group_keys[0]}_mean value": m1,
                f"N_{group_keys[1]}_mean value": m2   }
            
            if len(d1) > 2 and len(d2) > 2:
                _, p_val = stats.mannwhitneyu(d1, d2, alternative='two-sided')
                metadata["Statistics"][r_key][stat_key][f"{group_keys[0]}_vs_{group_keys[1]}_pval"] = float(p_val)
                
                star = get_asterisks(p_val)
                if star and star != 'ns':
                    local_max = max(np.max(d1), np.max(d2))
                    bracket_roof = local_max + (y_range * 0.05)
                    
                    add_stat_annotation_two_sided(ax, d1, d2, x1, x2, bracket_roof, ttest=0, paired=0)

        # --- SUBPLOT AESTHETICS ---
        #ax.set_title(r_title , pad=3)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, fontsize=6)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.5)
        ax.tick_params(axis='x', length=2, pad=1)
        
        # Shared Y-Axis Logic & Subtle Grid Boundaries
        if ax_idx == 0:
            ax.set_ylabel(y_label, labelpad=1)
            ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
            ax.tick_params(axis='y', length=2, pad=1)
            ax.spines['left'].set_color('black')
            ax.spines['left'].set_linewidth(0.5)
        else:
            ax.set_ylabel('')
            ax.set_yticks([])
            ax.tick_params(axis='y', length=0)
            # The subtle structural boundary for the 2nd and 3rd panels
            ax.spines['left'].set_visible(True)
            ax.spines['left'].set_color('#E0E0E0') # Very faint gray 
            ax.spines['left'].set_linewidth(0.5)

    # --- 5. UNIFIED LEGEND ---
    l_g1 = mlines.Line2D([], [], color=colors[0], marker='o', linestyle='None', markersize=3)
    l_g2 = mlines.Line2D([], [], color=colors[1], marker='o', linestyle='None', markersize=3)
    
    # Place legend in the first panel, upper left
    #axes[0].legend(handles=[l_g1, l_g2], labels=[group_keys[0], group_keys[1]],
    #               frameon=False, loc='upper left', handletextpad=0.2, borderpad=0)

    # --- 6. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    
    # Slight w_pad ensures the text labels don't crash into the faint boundaries
    fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4.8 PV similarity comparison for co_cs_us cells

In [ ]:
def extract_PETH_first_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        data_tmps = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_tmps.append(tmp[:, bin_s:bin_e]) #[:, bin_s:bin_e].mean(axis=0))        
            if trial_idx ==0:
                if len(data_tmps) >0:
                    data_trials.append(data_tmps[0])
                else:
                    data_trials.append(np.nan)
            else:   
                if len(data_tmps) > 0:
                    data_trials_extract = np.concatenate(data_tmps, axis=0)
                    data_trials.append(data_trials_extract)
                else:
                    data_trials.append(np.nan)                
        data_group.append(data_trials)
    return data_group
    
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 20 # 10s
post_du1 = 3
post_du2 = 6 #post shock  use 20s
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)), 
    'pus_2': (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

width_mm = 40  # 
height_mm = 30 #  

resp_name = 'intersec_tone_shock'
target_epoch ='us'
compare_epoch = 'cs'
bin_s = 0
bin_e = int((20+20+20+40)*fs)
ds_groups = {}
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)

    ds_groups[group_keys[i]] = extract_PETH_first_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=6)
   
plot_cross_epoch_reactivation(ds_groups, target_epoch, compare_epoch, epochs, group_keys, width_mm, height_mm, colors_beh_i, dpath_plot, 
                              f'04_8_Reactivation similarity with PV (clip 0)', flag_PV=1)
print('All finished************') 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def plot_cross_epoch_reactivation(ds_groups, target_epoch, compare_epoch, epochs, group_keys, width_mm, height_mm, colors, output_path, title, flag_PV=1, min_cells=4):
    """
    Plots categorized Cosine Similarity (Initial, Early, Late) between two epochs for 2 groups.
    Includes denoising: Animals with fewer than `min_cells` are excluded.
    
    Parameters:
    - ds_groups: Dict containing animal lists for each group.
      Structure: ds_groups[group_key] = [animal_1, animal_2, ...]
      Where animal_x = [trial_1_array, trial_2_array, ...] (arrays are n_cells x n_bins, already pooled)
    - flag_PV: 0 (Raw Mean), 1 (ReLU Mean)
    """
    set_pub_style()
    
    # Conditions: [Name, Target Trial Number (1-indexed)]
    conditions = [
        ('Initial', 1), # Trial 1
        ('Early', 2),   # First 2 trials (Union pooled at index 1)
        ('Late', 6)     # All 6 trials (Union pooled at index 5)
    ]
    x_labels = ['Initial\n(T1)', 'Early\n(T1-2)', 'All\n(T1-6)']
    
    t_start, t_end = epochs[target_epoch]
    c_start, c_end = epochs[compare_epoch]
    
    # --- 1. DATA EXTRACTION & DENOISING ---
    master_data = {}
    
    for g_key in group_keys:
        animal_list = ds_groups.get(g_key, [])
        n_animals = len(animal_list)
        
        metric_matrix = np.full((len(conditions), n_animals), np.nan)
        
        for a_idx, animal_trials in enumerate(animal_list):
            for c_idx, (c_name, t_num) in enumerate(conditions):
                
                t_idx = t_num - 1 # Convert 1-indexed to 0-indexed list index
                
                if t_idx < len(animal_trials):
                    trial_data = animal_trials[t_idx]
                    
                    if isinstance(trial_data, np.ndarray) and trial_data.ndim == 2:
                        # Apply Denoising Threshold
                        if trial_data.shape[0] >= min_cells:
                            
                            # Slice epochs
                            t_data = trial_data[:, t_start:t_end]
                            c_data = trial_data[:, c_start:c_end]
                            
                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore", category=RuntimeWarning)
                                
                                # PV Logic
                                if flag_PV == 0:
                                    pv_target = np.nanmean(t_data, axis=1)
                                    pv_compare = np.nanmean(c_data, axis=1)
                                elif flag_PV == 1:
                                    pv_target = np.nanmean(np.maximum(t_data, 0), axis=1)
                                    pv_compare = np.nanmean(np.maximum(c_data, 0), axis=1)
                                else:
                                    raise ValueError("flag_PV must be 0 or 1")
                                    
                            # Reshape & clean for sklearn
                            pv_target = np.nan_to_num(pv_target).reshape(1, -1)
                            pv_compare = np.nan_to_num(pv_compare).reshape(1, -1)
                            
                            # Calculate Similarity (Skip if vectors are entirely zero to avoid noisy 0.0 scores)
                            if np.sum(np.abs(pv_target)) > 0 and np.sum(np.abs(pv_compare)) > 0:
                                metric_matrix[c_idx, a_idx] = cosine_similarity(pv_target, pv_compare)[0, 0]
                                
        master_data[g_key] = metric_matrix

    # --- 2. GLOBAL Y-AXIS SCALING ---
    all_vals = []
    for g_key in group_keys:
        matrix = master_data[g_key]
        all_vals.extend(matrix[~np.isnan(matrix)])
        
    if len(all_vals) == 0:
        raise ValueError(f"No valid data found after filtering (min_cells={min_cells}).")
        
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min if global_max != global_min else 1.0
    
    # Cosine similarity typically shouldn't drop far below 0 with ReLU data
    y_bottom = max(-0.1, global_min - (y_range * 0.1))
    y_top = min(1.15, global_max + (y_range * 0.25))

    # --- 3. CANVAS SETUP ---
    # Single plot: width 40mm, height 30mm
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    x_positions = np.array([1, 2, 3])
    off_1, off_2 = -0.15, 0.15  
    
    c_dark = colors[0]
    c_light = lighten_color(c_dark, amount=0.4)
    i_dark = colors[1]
    i_light = lighten_color(i_dark, amount=0.4)

    g1_matrix = master_data[group_keys[0]]
    g2_matrix = master_data[group_keys[1]]

    # --- 4. PLOTTING LOOP ---
    for c_idx, (c_name, _) in enumerate(conditions):
        d1 = g1_matrix[c_idx, :]
        d2 = g2_matrix[c_idx, :]
        
        d1 = d1[~np.isnan(d1)]
        d2 = d2[~np.isnan(d2)]
        
        # Stats
        m1 = np.mean(d1) if len(d1) > 0 else np.nan
        s1 = stats.sem(d1) if len(d1) > 1 else np.nan
        m2 = np.mean(d2) if len(d2) > 0 else np.nan
        s2 = stats.sem(d2) if len(d2) > 1 else np.nan
        
        x_base = x_positions[c_idx]
        x1 = x_base + off_1
        x2 = x_base + off_2
        
        # Scatter Dots
        jitter = 0.06
        x1_scatter = x1 + np.random.uniform(-jitter, jitter, size=len(d1))
        x2_scatter = x2 + np.random.uniform(-jitter, jitter, size=len(d2))
        
        ax.scatter(x1_scatter, d1, s=2.5, color=c_light, alpha=0.9, edgecolors='none', zorder=1)
        ax.scatter(x2_scatter, d2, s=2.5, color=i_light, alpha=0.9, edgecolors='none', zorder=1)
        
        # Error Bars & Mean (Discrete, no connecting lines)
        if not np.isnan(m1):
            ax.errorbar(x1, m1, yerr=s1, fmt='o', color=c_dark, elinewidth=0.75, capsize=0, 
                        markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=3)
        if not np.isnan(m2):
            ax.errorbar(x2, m2, yerr=s2, fmt='o', color=i_dark, elinewidth=0.75, capsize=0, 
                        markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=3)
        
        # Independent Significance Testing (CA1-C vs CA1-I)
        metadata["Statistics"][c_name] = {
            f"N_{group_keys[0]}": len(d1),
            f"N_{group_keys[1]}": len(d2),
            f"N_{group_keys[0]}_mean": m1,
            f"N_{group_keys[1]}_mean": m2
        }
        
        if len(d1) > 2 and len(d2) > 2:
            _, p_val = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            metadata["Statistics"][c_name][f"{group_keys[0]}_vs_{group_keys[1]}_pval"] = float(p_val)
            
            star = get_asterisks(p_val)
            if star and star != 'ns':
                local_max = max(np.max(d1), np.max(d2))
                bracket_roof = local_max + (y_range * 0.05)
                add_stat_annotation_two_sided(ax, d1, d2, x1, x2, bracket_roof, ttest=0, paired=0)

    # --- 5. ZERO REFERENCE & AESTHETICS ---
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.75, alpha=0.5, zorder=0)
    
    #ax.set_title(f"PV_CS vs PV_US", pad=4)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=6)
    ax.set_ylabel('Cosine similarity\n(Reactivation)', labelpad=1)
    
    ax.set_ylim(y_bottom, y_top)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Legend
    l_c = mlines.Line2D([], [], color=c_dark, marker='o', linestyle='None', markersize=3)
    l_i = mlines.Line2D([], [], color=i_dark, marker='o', linestyle='None', markersize=3)
    
    # --- 6. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

# 5. Trace cells related 

## 5.1 Trace cell raw data plotting-heatmap

In [ ]:
group_name = ['02.CA1-C'] 
group_keys = ['CA1-C' ] 
group_size = len(group_name)

resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']

test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 50  # 
height_mm = 60 #   
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US': 9}
base_du = 20
base_plot_start = ((base_du - epochs['Base'])* fs)
base_plot_end = ((base_du +40 + 40)* fs)
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    dpath_cal_bin = os.path.join(dpath_cal_group, test_algori_data)

    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)    # (n_cells, n_trials)
    
    dict_group_cal = {}
    for key in list(dict_group_response.keys()):
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_cal_bin, f"{key}_Cal_bin_trial.nc"))  
        dict_group_cal[key] = Sig_bin_trial['Sig_each_trial'].values[:, :, base_plot_start:base_plot_end]
    plot_trace_subgroup_heatmap(dict_group_response, dict_group_cal, epochs, fs, width_mm, height_mm, dpath_plot, 
                            f'05_1_trace cell activity heatmap-trial 1-{group_keys[i]}')
print('All finished************')      

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
def plot_trace_subgroup_heatmap(dict_group_response, dict_group_cal, epochs, fs, width_mm, height_mm, output_path, title):
    """
    Plots a highly optimized, subgroup-sorted heatmap of raw Z-scored calcium traces
    for Trial 1 trace cells pooled across all animals in the group.
    
    Subgroups (Top to Bottom):
    1. trace & US & pUS 
    2. trace & pUS 
    3. trace & US 
    4. trace Only 
    """
    set_pub_style()    
    # 1. EXTRACT AND POOL DATA FOR TRIAL 1
    t_idx = 0 
    
    pooled_calcium = []
    pooled_flags = []
    
    for animal in dict_group_response.keys():
        flags = dict_group_response[animal]
        cal_data = dict_group_cal[animal] # Shape: (trials, cells, bins)
        
        # Get Trial 1 calcium traces for this animal
        cal_t1 = cal_data[t_idx] 
        
        # Get Trial 1 flags (Transpose so shape is (cells,))
        trace_t1 = np.array(flags['trace']).astype(bool).T[t_idx]
        cs_t1 = np.array(flags['cs']).astype(bool).T[t_idx]
        us_t1 = np.array(flags['us']).astype(bool).T[t_idx]
        # only care about cells that are trace-responsive
        trace_indices = np.where(trace_t1)[0]
        
        for idx in trace_indices:
            pooled_calcium.append(cal_t1[idx])
            pooled_flags.append({
                'cs': cs_t1[idx],
                'us': us_t1[idx]
            })

    pooled_calcium = np.array(pooled_calcium)
    if len(pooled_calcium) == 0:
        raise ValueError("No trace responsive cells found in Trial 1.")

    # 2. ASSIGN SUBGROUPS
    subgroup_ids = np.zeros(len(pooled_flags), dtype=int)
    for i, flag in enumerate(pooled_flags):
        if flag['us'] and flag['cs']:
            subgroup_ids[i] = 0
        elif flag['us'] and not flag['cs']:
            subgroup_ids[i] = 1
        elif not flag['us'] and flag['cs']:
            subgroup_ids[i] = 2
        else:
            subgroup_ids[i] = 3

    # 3. INTERNAL PEAK SORTING
    t_base_bins = int(epochs.get('Base', 3) * fs)
    t_cs_bins = int(epochs.get('CS', 20) * fs)
    t_trace_bins = int(epochs.get('Trace', 20) * fs)
    trace_start = t_base_bins + t_cs_bins
    trace_end = t_base_bins + t_cs_bins + t_trace_bins
    
    sorted_calcium = []
    subgroup_boundaries = [0] 
    
    for g_id in range(4):
        group_indices = np.where(subgroup_ids == g_id)[0]
        if len(group_indices) == 0:
            continue
            
        group_cal = pooled_calcium[group_indices]
        
        trace_epoch_data = group_cal[:, trace_start:trace_end]
        peak_times = np.argmax(trace_epoch_data, axis=1)
        
        sort_order = np.argsort(peak_times)
        sorted_group_cal = group_cal[sort_order]
        
        sorted_calcium.append(sorted_group_cal)
        subgroup_boundaries.append(subgroup_boundaries[-1] + len(group_indices))

    final_matrix = np.vstack(sorted_calcium)

    # 4. PLOTTING THE HEATMAP
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Calculate time boundaries in seconds to map the X-axis accurately
    base_sec = epochs.get('Base', 3)
    total_sec = final_matrix.shape[1] / fs
    x_start = -base_sec
    x_end = total_sec - base_sec
    n_cells = final_matrix.shape[0]

    # Map the array mathematically to [Time Start, Time End, Bottom Cell, Top Cell]
    im = ax.imshow(final_matrix, aspect='auto', cmap='coolwarm', 
                   vmin=-3, vmax=3, interpolation='nearest', #vmin=-0.5, vmax=3.5
                   extent=[x_start, x_end, n_cells, 0])             
    # 5. AESTHETICS & EPOCH ANNOTATIONS
    # Draw vertical lines for epoch transitions using REAL time (seconds)
    current_time_sec = -base_sec
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time_sec += ep_dur
            continue
        ax.axvline(current_time_sec, color='white', linestyle='--', linewidth=0.5, alpha=0.8)
        current_time_sec += ep_dur
        if ep_name == 'P_US':
            ax.axvline(current_time_sec, color='white', linestyle='--', linewidth=0.5, alpha=0.8)

    # Draw horizontal lines to separate the 4 subgroups
    for boundary in subgroup_boundaries[1:-1]:
        ax.axhline(boundary, color='black', linestyle='-', linewidth=0.5, alpha=0.8)

    # Formatting Axes
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel(f'Trace cells',  labelpad=1)
    
    # Set requested intervals: X every 20s, Y every 100 cells
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(100))
    
    ax.tick_params(axis='both',length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Tiny Colorbar
    #cax = inset_axes(ax, width="5%", height="70%", loc='lower right', 
    #             bbox_to_anchor=(0.1, 0, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    #cbar = plt.colorbar(im, cax=cax)
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.5, anchor=(1.0, 0.05))
    cbar.set_label('Z-Score', labelpad=1)
    cbar.ax.tick_params(labelsize=4.5, length=1, pad=1)
    cbar.outline.set_linewidth(0.5)

    # 6. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=600, transparent=False)
    plt.close()

## 5.2  Area-Proportional Euler diagrams for Trace cells

In [ ]:
group_name = ['02.CA1-C']
group_keys = ['CA1-C'] 
group_size = len(group_name)

resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']
test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'

width_mm = 30  # 
height_mm = 30 #   

for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    
    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)     # shape (n_cells, n_trials)

    trial_idx = 0
    plot_engram_euler_diagram_trace(dict_group_response, trial_idx, width_mm, height_mm, dpath_plot, f'05_2_Area-Proportional Euler diagrams-trial-1_for Trace cells-{group_keys[i]}')

print('All finished************')      

In [ ]:
from matplotlib_venn import venn3
def plot_engram_euler_diagram_trace(dict_group_response, target_trial=0, width_mm=30, height_mm=30, output_path="./", title="Euler_Trial_1"):
    """
    Calculates exact ensemble intersections across all animals for a specific trial
    and plots an Area-Proportional Euler Diagram (Venn).
    target_trial: 0-indexed (0 = Trial 1).
    CS Only: #4091cf (Blue)
    US Only: #e1703c (Orange)
    Trace Only: #8cba54 (Green)
    co_cs_us: #a08085
    co_cs_trace: #66a591
    co_us_trace: #b69548
    co_cs_us_trace: #8f9375
    """
    set_pub_style()
    animal_keys = list(dict_group_response.keys())
    
    # Initialize subset counters
    # The 7 subsets for 3 circles: '100', '010', '110', '001', '101', '011', '111'
    # Format: CS, US, Trace
    subsets = {'100': 0, '010': 0, '001': 0, 
               '110': 0, '101': 0, '011': 0, 
               '111': 0}
               
    total_eligible_cells = 0
    total_cs = 0
    total_us = 0
    total_trace = 0
    # Aggregate cells across all animals for the specified trial
    for animal in animal_keys:
        flags = dict_group_response[animal]
        
        cs_t = np.array(flags['cs']).astype(bool).T[target_trial]
        us_t = np.array(flags['us']).astype(bool).T[target_trial]
        trace_t = np.array(flags['trace']).astype(bool).T[target_trial]
        
        total_eligible_cells += len(cs_t)      
        # Track total counts for metadata
        total_cs += np.sum(cs_t)
        total_us += np.sum(us_t)
        total_trace += np.sum(trace_t)

        
        # Calculate exactly which subset each cell belongs to
        # True=1, False=0. A cell that is only CS is '100'. A cell in all three is '111'.
        subsets['100'] += np.sum(cs_t & ~us_t & ~trace_t)
        subsets['010'] += np.sum(~cs_t & us_t & ~trace_t)
        subsets['001'] += np.sum(~cs_t & ~us_t & trace_t)
        
        subsets['110'] += np.sum(cs_t & us_t & ~trace_t)
        subsets['101'] += np.sum(cs_t & ~us_t & trace_t)
        subsets['011'] += np.sum(~cs_t & us_t & trace_t)
        
        subsets['111'] += np.sum(cs_t & us_t & trace_t)

    # Compile Metadata
    metadata = {
        "Figure_Title": title,
        "Trial_Index": target_trial,
        "Total_Eligible_Cells": int(total_eligible_cells),
        "Total_Trace_Cells": int(total_trace),
        "Total_US_Cells": int(total_us),
        "Total_CS_Cells": int(total_cs),
        "Subsets_Intersection_Counts": {
            "CS_Only": int(subsets['100']),
            "US_Only": int(subsets['010']),
            "Trace_Only": int(subsets['001']),
            "CS_and_US": int(subsets['110']),
            "CS_and_pUS": int(subsets['101']),
            "US_and_Trace": int(subsets['011']),
            "CS_and_US_and_Trace": int(subsets['111'])
        }
    }

    # --- PLOTTING ---
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Professional Colors: CS (Blue), US (Orange), pUS (Green)
    v = venn3(subsets=subsets, set_labels=('CS', 'US', 'Trace'), ax=ax, 
              set_colors=('#4091cf', '#e1703c', '#8cba54'), alpha=0.6)

    # Formatting: Convert integer counts to Percentages of Total Cells
    for subset_id in subsets.keys():
        patch = v.get_patch_by_id(subset_id)
        label = v.get_label_by_id(subset_id)
        if label:
            count = subsets[subset_id]
            # Avoid division by zero
            percentage = (count / total_eligible_cells) * 100 if total_eligible_cells > 0 else 0
            
            # If percentage is too small, hide the text to prevent clutter
            if percentage > 0.5:
                label.set_text(f"{percentage:.2f}%")
                label.set_fontsize(5)
            else:
                label.set_text("")
                
            # Optional: Add subtle borders to the circles for a crisp look
            if patch:
                patch.set_edgecolor('white')
                patch.set_linewidth(0.5)

    # Format circle labels (CS, US, pUS)
    if v.set_labels:
        for text in v.set_labels:
            if text:
                text.set_fontsize(6)
                text.set_fontweight('bold')

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title)
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=300, transparent=True)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## supp_5.3 Conditional probability test for Trace cell bridging

In [ ]:
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I']
group_size = len(group_name)

test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'

width_mm = 50  # 
height_mm = 30 #   
targets = ['us', 'pus']

for idx, target in enumerate(targets):
    dict_group = {}
    for i in range(group_size):    
        dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
        dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
        
        print(dpath_cal_group)
        # Resp cell info
        with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
                dict_group_response = pickle.load(f)    
        dict_group[group_keys[i]] = dict_group_response

    # PLotting
    plot_conditional_probability_denoised_trace(dict_group, target, 'Cond. prob. (CS react.)', width_mm, height_mm, colors_beh_i, dpath_plot, 
                                               f'sup_05_3_{idx+1}_Conditional Probability of CS reactivation in epoch {target}')
print('All finished************')      

In [ ]:
def plot_conditional_probability_denoised_trace(dict_group, target, y_label, width_mm, height_mm, colors, output_path, title, 
                                                    min_base_cells=10, min_denom_cells=3, apply_laplace_smoothing=True):
    """
    Calculates denoised conditional reactivation probabilities (P1 vs P2) for a single target,
    plotted side-by-side for 2 experimental groups across 'Trial 1' and 'All Trials'.
    Strictly follows top-tier low-ink typography standards (no bolding).
    """
    set_pub_style()
    group_keys = list(dict_group.keys())
    if len(group_keys) < 2:
        raise ValueError("dict_group must contain exactly two group keys (e.g., CA1-C and CA1-I).")
        
    g1_key, g2_key = group_keys[0], group_keys[1]
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Target": target, "Statistics": {}}
    
    # --- 1. DATA EXTRACTION ---
    g1_p1_t1, g1_p2_t1, g1_p1_all, g1_p2_all = extract_cond_probs_for_group(
        dict_group[g1_key], target, min_base_cells, min_denom_cells, apply_laplace_smoothing)
    g2_p1_t1, g2_p2_t1, g2_p1_all, g2_p2_all = extract_cond_probs_for_group(
        dict_group[g2_key], target, min_base_cells, min_denom_cells, apply_laplace_smoothing)
        
    # --- 2. COLOR & LAYOUT CONFIGURATION ---
    c_dark = colors[0]       
    c_light = lighten_color(c_dark, amount=0.4) 
    
    i_dark = colors[1]       
    i_light = lighten_color(i_dark, amount=0.4)
    
    # Mapping: d1 = P2 (~Trace baseline), d2 = P1 (Trace integration)
    plot_configs = [
        {'name': f'{g1_key}_Trial1', 'd1': g1_p2_t1,  'd2': g1_p1_t1,  'x': 1, 'c_light': c_light, 'c_dark': c_dark},
        {'name': f'{g1_key}_All',    'd1': g1_p2_all, 'd2': g1_p1_all, 'x': 2, 'c_light': c_light, 'c_dark': c_dark},
        {'name': f'{g2_key}_Trial1', 'd1': g2_p2_t1,  'd2': g2_p1_t1,  'x': 3, 'c_light': i_light, 'c_dark': i_dark},
        {'name': f'{g2_key}_All',    'd1': g2_p2_all, 'd2': g2_p1_all, 'x': 4, 'c_light': i_light, 'c_dark': i_dark},
    ]

    # --- 3. GLOBAL SCALING ---
    all_vals = []
    for cfg in plot_configs:
        all_vals.extend(cfg['d1'][~np.isnan(cfg['d1'])])
        all_vals.extend(cfg['d2'][~np.isnan(cfg['d2'])])
        
    global_max = max(all_vals) if len(all_vals) > 0 else 1.0
    global_min = min(all_vals) if len(all_vals) > 0 else 0.0
    y_range = global_max - global_min if global_max != global_min else 1.0
    
    ax.set_ylim(max(0, global_min - (y_range * 0.05)), global_max + (y_range * 0.35))

    # --- 4. PLOTTING LOOP ---
    off1, off2 = -0.15, 0.15
    
    for cfg in plot_configs:
        d1_raw, d2_raw = cfg['d1'], cfg['d2']
        valid_mask = ~np.isnan(d1_raw) & ~np.isnan(d2_raw)
        d1 = d1_raw[valid_mask]
        d2 = d2_raw[valid_mask]
        
        # Metadata Logging
        metadata["Statistics"][cfg['name']] = {
            "N_animals_survived": int(np.sum(valid_mask)),
            "Mean_P2_no_trace": float(np.mean(d1)) if len(d1) > 0 else None,
            "SEM_P2_no_trace": float(stats.sem(d1)) if len(d1) > 1 else None,
            "Mean_P1_trace": float(np.mean(d2)) if len(d2) > 0 else None,
            "SEM_P1_trace": float(stats.sem(d2)) if len(d2) > 1 else None,
        }
        
        if len(d1) == 0: continue
            
        x1 = cfg['x'] + off1
        x2 = cfg['x'] + off2
        
        # Paired Lines
        for i in range(len(d1)):
            ax.plot([x1, x2], [d1[i], d2[i]], color='gray', alpha=0.3, lw=0.5, zorder=0)
            
        # Scatter Dots
        ax.scatter(np.full(len(d1), x1), d1, s=2.5, color=cfg['c_light'], alpha=0.9, edgecolors='none', zorder=2)
        ax.scatter(np.full(len(d2), x2), d2, s=2.5, color=cfg['c_dark'], alpha=0.9, edgecolors='none', zorder=2)
        
        # Means and Error Bars
        m1, s1 = np.mean(d1), stats.sem(d1)
        m2, s2 = np.mean(d2), stats.sem(d2)
        
        ax.plot([x1, x2], [m1, m2], color='black', lw=1.0, zorder=1)
        ax.errorbar(x1, m1, yerr=s1, fmt='o', color=cfg['c_light'], elinewidth=0.75, capsize=0, 
                    markersize=4, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
        ax.errorbar(x2, m2, yerr=s2, fmt='o', color=cfg['c_dark'], elinewidth=0.75, capsize=0, 
                    markersize=4, markeredgecolor='white', markeredgewidth=0.25, zorder=4)
        
        # Paired Wilcoxon Testing
        if len(d1) > 3 and not np.all(d1 == d2):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                _, p_val = stats.wilcoxon(d1, d2, alternative='two-sided')
                metadata["Statistics"][cfg['name']]["P2_vs_P1_pval"] = float(p_val)      
                
                star = get_asterisks(p_val)
                if star and star != 'ns':
                    local_max = max(np.max(d1), np.max(d2))
                    bracket_roof = local_max + (y_range * 0.05)
                    add_stat_annotation_two_sided(ax, d1, d2, x1, x2, bracket_roof, ttest=0, paired=1)

    # --- 5. TOP-TIER TYPOGRAPHY & FORMATTING (No Bold) ---
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xticklabels(['Trial 1', 'All trials', 'Trial 1', 'All trials'], fontsize=6) # Regular weight
    ax.set_ylabel(y_label, labelpad=0.1) # Regular weight
    
    # Subtle Structural Separator & Clean Group Headers
    ax.axvline(2.5, color='gray', linestyle=':', lw=0.5, alpha=0.5)
    
    header_y = global_max + (y_range * 0.25)
    #ax.text(1.5, header_y, g1_key, ha='center', va='bottom') # Regular weight
    #ax.text(3.5, header_y, g2_key, ha='center', va='bottom') # Regular weight
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Minimalist Tuple Legend
    l_light_c = mlines.Line2D([], [], color=c_light, marker='o', linestyle='None', markersize=3)
    l_light_i = mlines.Line2D([], [], color=i_light, marker='o', linestyle='None', markersize=3)
    l_dark_c = mlines.Line2D([], [], color=c_dark, marker='o', linestyle='None', markersize=3)
    l_dark_i = mlines.Line2D([], [], color=i_dark, marker='o', linestyle='None', markersize=3)
    
    ax.legend(handles=[(l_light_c, l_light_i), (l_dark_c, l_dark_i)], 
              labels=['P$_2$ (~Trace)', 'P$_1$ (Trace)'],
              handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
              frameon=False, loc='upper left', handletextpad=0.5, borderpad=0)

    # --- 6. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def extract_cond_probs_for_group(dict_group_response, target, min_base_cells, min_denom_cells, apply_laplace_smoothing):
    """Internal helper to calculate denoised P1 and P2 for Trial 1 and All Trials (Mean)."""
    animal_keys = list(dict_group_response.keys())
    p1_t1_list, p2_t1_list = [], []
    p1_all_list, p2_all_list = [], []
    
    for animal in animal_keys:
        flags = dict_group_response[animal]
        cs_flags = np.array(flags['cs']).astype(bool).T
        trace_flags = np.array(flags['trace']).astype(bool).T
        n_trials = cs_flags.shape[0]
        
        # Target resolution
        if target == 'pus':
            pus1 = np.array(flags['pus_1']).astype(bool).T
            pus2 = np.array(flags['pus_2']).astype(bool).T
            tgt_flags = pus1 | pus2
        else:
            tgt_flags = np.array(flags[target]).astype(bool).T
            
        p1_trials = np.full(n_trials, np.nan)
        p2_trials = np.full(n_trials, np.nan)
        
        for t in range(n_trials):
            cs_t = cs_flags[t]
            trace_t = trace_flags[t]
            tgt_t = tgt_flags[t]
            
            if np.sum(cs_t) < min_base_cells:
                continue
                
            cs_and_trace = cs_t & trace_t
            cs_not_trace = cs_t & ~trace_t
            
            n_cs_trace = np.sum(cs_and_trace)
            n_cs_not_trace = np.sum(cs_not_trace)
            
            if not apply_laplace_smoothing and (n_cs_trace < min_denom_cells or n_cs_not_trace < min_denom_cells):
                continue
                
            n_trace_tgt = np.sum(cs_and_trace & tgt_t)
            n_not_trace_tgt = np.sum(cs_not_trace & tgt_t)
            
            if apply_laplace_smoothing:
                p1_trials[t] = (n_trace_tgt + 1) / (n_cs_trace + 2)
                p2_trials[t] = (n_not_trace_tgt + 1) / (n_cs_not_trace + 2)
            else:
                p1_trials[t] = n_trace_tgt / n_cs_trace
                p2_trials[t] = n_not_trace_tgt / n_cs_not_trace
                
        # Extract Trial 1 (index 0)
        p1_t1_list.append(p1_trials[0])
        p2_t1_list.append(p2_trials[0])
        
        # Extract All Trials longitudinal average
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            p1_all_list.append(np.nanmean(p1_trials))
            p2_all_list.append(np.nanmean(p2_trials))
            
    return np.array(p1_t1_list), np.array(p2_t1_list), np.array(p1_all_list), np.array(p2_all_list)

## supp_5.4 CDF of peak latency co-resp cells reactivation in us, pus epochs

In [ ]:
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_data_dir = 'post_02_2_resp_cal_new_z'

resp_names = ['tone', 'trace'] #
resp_g1 = ['tone', 'trace'] # 
resp_g2 = ['shock', 'p_shock1', 'p_shock2'] 


base_du = 20 # 20s
post_du1 = 3 #
post_du2 = 6
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)),
    'pus_2': (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs)),
    'pus': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}
"""
  CS Only: #4091cf (Blue)
    US Only: #e1703c (Orange)
    Trace Only: #8cba54 (Green)
    co_cs_us: #a08085
    co_cs_trace: #66a591
    co_us_trace: #b69548
    co_cs_us_trace: #8f937
"""

height_mm = 30 #  
colors = ['#b69548', '#a08085', '#e1703c'] 

# For us epoch
resp_names_us = ['intersec_trace_shock', 'intersec_tone_shock', 'shock_only']
co_resp_names_1 = [r'US$_{\text{Trace}}$', r'US$_{\text{CS}}$', r'US$_{\text{Only}}$']
target_epoch_1 = 'us'

resp_names_pus = ['intersec_trace_p_shock', 'intersec_tone_p_shock', 'intersec_shock_p_shock']
co_resp_names_2 = [r'pUS$_{\text{Trace}}$',  r'pUS$_{\text{CS}}$', r'pUS$_{\text{US}}$', ]
target_epoch_2 = 'pus'
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    print(group_name[i])
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
    
    ls_data_1 = []
    for resp_n in resp_names_us:
        ls_data_1.append(cal_tmp[resp_n])
    ls_data_2 = []
    for resp_n in resp_names_pus:
        ls_data_2.append(cal_tmp[resp_n])
    dict_data = {target_epoch_1:ls_data_1,
                target_epoch_2:ls_data_2}
    if i==0:
        width_mm = 60  # 
    else:
        width_mm = 50  # 
    plot_cdf_timing_co_resp_cells(dict_data,co_resp_names_1, target_epoch_1, co_resp_names_2, target_epoch_2, epochs,  width_mm, height_mm, colors, 
                                      dpath_plot, f'sup_05_4_CDF of peak latency_of co-resp cells in us and pus epoch-{group_name[i]}')


print('All finished************')    

In [ ]:
def plot_cdf_timing_co_resp_cells(dict_data, co_resp_names_1, target_epoch_1, co_resp_names_2, target_epoch_2, epochs, width_mm, height_mm, colors, output_path, title, fs=5.0):
    """
    Generates a 1x2 CDF micro-panel comparing peak latencies across subgroups for ALL trials.
    Panel 1: us epoch (Width ratio 2)
    Panel 2: pus epoch (Width ratio 3)
    Includes Kruskal-Wallis + Post-hoc K-S Test (Bonferroni corrected against Group 0).
    """
    set_pub_style()
    
    plot_configs = [
        {'epoch': target_epoch_1, 'names': co_resp_names_1},
        {'epoch': target_epoch_2, 'names': co_resp_names_2}
    ]
    
    # --- 1. DATA EXTRACTION ---
    # master_data[panel_idx][group_idx] = array of peak latencies
    master_data = [] 
    
    for cfg in plot_configs:
        ep = cfg['epoch']
        t_start, t_end = epochs[ep]
        n_groups = len(cfg['names'])
        ep_data = []
        
        for g_idx in range(n_groups):
            ensemble_peaks = []
            # Access data from the dict using the epoch key
            for animal_trials in dict_data[ep][g_idx]:
                for t_idx in range(0, 6):  #  'All trials' (0 through 5)
                    if t_idx < len(animal_trials):
                        trial_data = animal_trials[t_idx]
                        
                        if isinstance(trial_data, np.ndarray) and trial_data.ndim == 2:
                            zoomed_data = trial_data[:, t_start:t_end]
                            if zoomed_data.shape[0] > 0:
                                # Calculate latency relative to the start of the epoch
                                peak_bins = np.argmax(zoomed_data, axis=1)
                                peak_sec = peak_bins / fs
                                ensemble_peaks.extend(peak_sec)
                                
            ep_data.append(np.array(ensemble_peaks))
        master_data.append(ep_data)

    # --- 2. CANVAS SETUP ---
    # Width ratios 2:3
    fig, axes = plt.subplots(1, 2, figsize=(width_mm/ 25.4, height_mm / 25.4), 
                             gridspec_kw={'width_ratios': [2, 3]}, 
                             layout='constrained', sharey=True)
                             
    metadata = {"Figure_Title": title, "Panels": {}}

    # --- 3. PLOTTING & STATS LOOP ---
    for c_idx, cfg in enumerate(plot_configs):
        ax = axes[c_idx]
        ep_name = cfg['epoch']
        co_resp_names = cfg['names']
        data_groups = master_data[c_idx]
        
        t_start, t_end = epochs[ep_name]
        time_window_sec = (t_end - t_start) / fs
        
        legend_labels = list(co_resp_names)
        metadata["Panels"][ep_name] = {"Cell_Counts": {}, "Statistics": {}}
        
        for i in range(len(co_resp_names)):
            metadata["Panels"][ep_name]["Cell_Counts"][co_resp_names[i]] = len(data_groups[i])

        # Statistical Testing (Omnibus + Post-Hoc vs Group 0)
        # Assumes the first group (index 0) is the control/baseline for KS testing
        if len(data_groups) >= 3 and all(len(d) > 2 for d in data_groups):
            stat_kw, p_kw = stats.kruskal(*data_groups)
            metadata["Panels"][ep_name]["Statistics"]["Kruskal_Wallis_pval"] = float(p_kw)
            
            if p_kw < 0.05:
                n_comparisons = len(data_groups) - 1 # Comparing all subsequent groups to G0
                
                for g_idx in range(1, len(data_groups)):
                    _, p_raw = stats.ks_2samp(data_groups[0], data_groups[g_idx])
                    p_adj = min(p_raw * n_comparisons, 1.0)
                    star = get_asterisks(p_adj)
                    metadata["Panels"][ep_name]["Statistics"][f"KS_{co_resp_names[0]}_vs_{co_resp_names[g_idx]}_adj_pval"] = float(p_adj)
                    
                    if star and star != 'ns':
                        legend_labels[g_idx] = f"{co_resp_names[g_idx]} {star}"

        # CDF Plotting
        for i in range(len(data_groups)):
            if len(data_groups[i]) > 0:
                sns.ecdfplot(data_groups[i], color=colors[i], ax=ax, linewidth=0.75, legend=False)

        # Subplot Aesthetics
        #ax.set_title(ep_name.upper(), pad=3)
        ax.set_xlabel('Peak latency (s)', labelpad=1)
        ax.set_xlim(0, time_window_sec)
        ax.set_ylim(0, 1.05)

        # Match user's specific axis tick logic safely
        if 'us' in ep_name.lower() and 'pus' not in ep_name.lower():
            ax.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
        else:
            ax.xaxis.set_major_locator(ticker.MultipleLocator(2.0))
            
        ax.tick_params(axis='x', length=2, pad=1)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.5)
        
        # Legend
        custom_lines = [Line2D([0], [0], color=colors[i], lw=1.0) for i in range(len(data_groups))]
        ax.legend(custom_lines, legend_labels, frameon=False, loc='lower right', 
                  handlelength=1.2, handletextpad=0.3, borderpad=0.1, fontsize=5)
        
        # Shared Y-Axis logic & Subtle internal spine
        if c_idx == 0:
            ax.set_ylabel('Cumulative fraction', labelpad=0.1)
            ax.yaxis.set_major_locator(ticker.MultipleLocator(0.25))
            ax.tick_params(axis='y', length=2, pad=1)
            ax.spines['left'].set_color('black')
            ax.spines['left'].set_linewidth(0.5) 
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', length=0)
            ax.spines['left'].set_visible(True)
            ax.spines['left'].set_color('#E0E0E0') # Subtle separator
            ax.spines['left'].set_linewidth(0.5)

    # --- 4. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 5.5 Co-resp cells PETH (co-trtace-us, co-trace-pus-1, co-trace-pus_2) animal-wise in CA1

In [ ]:
def extract_PETH_mean_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp) #[:, bin_s:bin_e].mean(axis=0))        
        if len(data_trials) > 0:
            data_trials = np.concatenate(data_trials, axis=0)
            data_trials = np.nanmean(data_trials, axis=0)[bin_s:bin_e]
            data_group.append(data_trials)
        #else:
         #   data_group.append(np.nan)
    return np.array(data_group)
    
group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 3 # 10s
post_du = 20 #post shock  use 20s
bin_s = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
bin_e = int((20+20+20+3+ post_du)*fs) # post-shock 17s

width_mm = 40  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

resp_names = ['intersec_trace_shock', 'intersec_trace_p_shock1', 'intersec_trace_p_shock2']#  'tone', 'trace',
resp_keys = ['co_trace_us', 'co_trace_pus_1', 'co_trace_pus_2']

for idx, resp_name in enumerate(resp_names):
    ds_groups = {}
    for i in range(group_size):    
        dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
        with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
            cal_tmp = pickle.load(f)
        ds_groups[group_keys[i]] = extract_PETH_mean_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=6)  

    # 1. plot of ALl 6 trials -Mean Z score 
    plot_trial_population_activity(ds_groups, 'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 
                                   f'05_5_{idx+1}_all 6 trials-Mean Z score-CA1-animal-wise_{resp_keys[idx]} resp cells')
    
print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    cal_trial : dict()--group name: (n_anmials, n_bins)
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 5.6 Co-resp cells Mean z score statictics (co-trace-us, co-trace-pus-1, co-trace-pus_2) animal-wise in CA1

In [ ]:
def extract_PETH_first_trials(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        data_tmps = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_tmps.append(tmp[:, bin_s:bin_e]) #[:, bin_s:bin_e].mean(axis=0))        
            if trial_idx ==0:
                if len(data_tmps) >0:
                    data_trials.append(data_tmps[0])
                else:
                    data_trials.append(np.nan)
            else:   
                if len(data_tmps) > 0:
                    data_trials_extract = np.concatenate(data_tmps, axis=0)
                    data_trials.append(data_trials_extract)
                else:
                    data_trials.append(np.nan)
                
        data_group.append(data_trials)
    return data_group
    
def extract_PETH_each_trial(cal_data, resp_name, bin_s, bin_e, trials_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trials_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp[:, bin_s:bin_e]) #[:, bin_s:bin_e].mean(axis=0))
            else:
                data_trials.append(np.nan)
        data_group.append(data_trials)
    return data_group   
   
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I']
group_size = len(group_name)

test_algori_data = 'post_02_2_resp_cal_new_z'

base_du = 20 # 10s
post_du1 = 3
post_du2 = 6 #post shock  use 20s
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)), 
    'pus_2': (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

resp_names = ['intersec_trace_shock', 'intersec_trace_p_shock1', 'intersec_trace_p_shock2']#  'tone', 'trace',
resp_keys = ['co_trace_us', 'co_trace_pus_1', 'co_trace_pus_2']
epoch_names = ['us', 'pus_1', 'pus_2']

# Merge all 3 co-resp groups
width_mm = 70  # 
height_mm = 30 #  

ds_groups = {}
for i in range(group_size):    
    
    dpath_data_group = os.path.join(dpath_cal_all, test_algori_data)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
    
    ds_resp_data = {}
    for idx, resp_name in enumerate(resp_names):
        bin_s, bin_e = epochs[epoch_names[idx]]
        ds_resp_data[resp_keys[idx]] = extract_PETH_first_trials(cal_tmp, resp_name, bin_s, bin_e, trials_num=6)
    ds_groups[group_keys[i]] = ds_resp_data
    
plot_mean_z_score_stat_animal_wise(ds_groups, 'Mean Z-Score', resp_keys, width_mm, height_mm, group_keys, colors_beh_i, dpath_plot, 
                       f'05_6_Mean Z score statistics of trace co-resp cells-CA1-animal-wise', min_base_cells=5)
print('All finished************') 

In [ ]:
def plot_mean_z_score_stat_animal_wise(ds_group, y_label, resp_keys, width_mm, height_mm, group_keys, colors, output_path, title, min_base_cells=3):
    """
    Plots categorized Mean Z-scores (Initial, Early, Late) for 3 co-response subgroups in a 1x3 grid.
    Includes denoising: Conditions with fewer than `min_base_cells` are excluded.
    
    Parameters:
    - ds_group: Dict containing animal lists for each group and subgroup.
      Structure: ds_group[group_key][resp_key] = [animal_1, animal_2, ...]
      Where animal_x = [trial_1_array, trial_2_array, ...] (arrays are n_cells x n_bins, already pooled)
    """
    set_pub_style()
    resp_titles = ['CS $\cap$ US', 'CS $\cap$ pUS$_1$', 'CS $\cap$ pUS$_2$']    
    # Conditions: [Name, Target Trial Number (1-indexed)]
    conditions = [
        ('Initial', 1), # Trial 1
        ('Early', 2),   # First 2 trials (Union pooled at index 1)
        ('Late', 6)     # All 6 trials (Union pooled at index 5)
    ]
    x_labels = ['Initial\n(T1)', 'Early\n(T1-2)', 'All\n(T1-6)']
    
    # --- 1. DATA EXTRACTION & DENOISING ---
    master_data = {r_key: {} for r_key in resp_keys}
    
    for r_key in resp_keys:
        for g_key in group_keys:
            animal_list = ds_group.get(g_key, {}).get(r_key, [])
            n_animals = len(animal_list)
            
            metric_matrix = np.full((len(conditions), n_animals), np.nan)
            
            for a_idx, animal_trials in enumerate(animal_list):
                for c_idx, (c_name, t_num) in enumerate(conditions):
                    
                    t_idx = t_num - 1 # Convert 1-indexed trial number to 0-indexed list index
                    
                    if t_idx < len(animal_trials):
                        trial_data = animal_trials[t_idx]                        
                        # Validate the array
                        if isinstance(trial_data, np.ndarray) and trial_data.ndim == 2:                          
                            # Apply Denoising Threshold
                            if trial_data.shape[0] >= min_base_cells:
                                with warnings.catch_warnings():
                                    warnings.simplefilter("ignore", category=RuntimeWarning)
                                    metric_matrix[c_idx, a_idx] = np.nanmean(trial_data)
                                
            master_data[r_key][g_key] = metric_matrix
    # --- 2. GLOBAL Y-AXIS SCALING ---
    all_vals = []
    for r_key in resp_keys:
        for g_key in group_keys:
            matrix = master_data[r_key][g_key]
            all_vals.extend(matrix[~np.isnan(matrix)])
            
    if len(all_vals) == 0:
        raise ValueError(f"No valid data found after filtering (min_base_cells={min_base_cells}).")
        
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min if global_max != global_min else 1.0
    
    shared_ymin = global_min - (y_range * 0.05)
    shared_ymax = global_max + (y_range * 0.25) # Headroom for brackets

    # --- 3. CANVAS SETUP (1x3 Facet Grid) ---
    fig, axes = plt.subplots(1, 3, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    x_positions = np.array([1, 2, 3])
    off_1, off_2 = -0.15, 0.15  

    # --- 4. PLOTTING LOOP ---
    for ax_idx, (ax, r_key, r_title) in enumerate(zip(axes, resp_keys, resp_titles)):
        metadata["Statistics"][r_key] = {}
        
        g1_matrix = master_data[r_key][group_keys[0]]
        g2_matrix = master_data[r_key][group_keys[1]]
        
        ax.set_ylim(shared_ymin, shared_ymax)
        ax.set_xlim(0.5, 3.5)
        
        for c_idx, (c_name, _) in enumerate(conditions):
            d1 = g1_matrix[c_idx, :]
            d2 = g2_matrix[c_idx, :]
            
            # Mask out NaNs
            d1 = d1[~np.isnan(d1)]
            d2 = d2[~np.isnan(d2)]
            
            # Stats
            m1 = np.mean(d1) if len(d1) > 0 else np.nan
            s1 = stats.sem(d1) if len(d1) > 1 else np.nan
            m2 = np.mean(d2) if len(d2) > 0 else np.nan
            s2 = stats.sem(d2) if len(d2) > 1 else np.nan
            
            x_base = x_positions[c_idx]
            x1 = x_base + off_1
            x2 = x_base + off_2
            
            # Scatter Dots
            jitter = 0.06
            x1_scatter = x1 + np.random.uniform(-jitter, jitter, size=len(d1))
            x2_scatter = x2 + np.random.uniform(-jitter, jitter, size=len(d2))
            
            ax.scatter(x1_scatter, d1, s=2.0, color=colors[0], alpha=0.4, edgecolors='none', zorder=1)
            ax.scatter(x2_scatter, d2, s=2.0, color=colors[1], alpha=0.4, edgecolors='none', zorder=1)
            
            # Error Bars & Mean
            if not np.isnan(m1):
                ax.errorbar(x1, m1, yerr=s1, fmt='o', color=colors[0], elinewidth=0.75, capsize=0, 
                            markersize=3.5, markeredgecolor='white', markeredgewidth=0.25, zorder=3)
            if not np.isnan(m2):
                ax.errorbar(x2, m2, yerr=s2, fmt='o', color=colors[1], elinewidth=0.75, capsize=0, 
                            markersize=3.5, markeredgecolor='white', markeredgewidth=0.25, zorder=3)
            
            # Significance Testing
            stat_key = f"{c_name}"
            metadata["Statistics"][r_key][stat_key] = {
                f"N_{group_keys[0]}": len(d1),
                f"N_{group_keys[1]}": len(d2)}
            
            if len(d1) > 2 and len(d2) > 2:
                _, p_val = stats.mannwhitneyu(d1, d2, alternative='two-sided')
                metadata["Statistics"][r_key][stat_key][f"{group_keys[0]}_vs_{group_keys[1]}_pval"] = float(p_val)
                
                star = get_asterisks(p_val)
                if star and star != 'ns':
                    local_max = max(np.max(d1), np.max(d2))
                    bracket_roof = local_max + (y_range * 0.05)
                    
                    add_stat_annotation_two_sided(ax, d1, d2, x1, x2, bracket_roof, ttest=0, paired=0)

        # --- SUBPLOT AESTHETICS ---
        #ax.set_title(r_title, pad=3)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, fontsize=6)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.5)
        ax.tick_params(axis='x', length=2, pad=1)
        
        # Shared Y-Axis Logic & Subtle Grid Boundaries
        if ax_idx == 0:
            ax.set_ylabel(y_label,  labelpad=1)
            ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
            ax.tick_params(axis='y', length=2, pad=1)
            ax.spines['left'].set_color('black')
            ax.spines['left'].set_linewidth(0.5)
        else:
            ax.set_ylabel('')
            ax.set_yticks([])
            ax.tick_params(axis='y', length=0)
            # The subtle structural boundary for the 2nd and 3rd panels
            ax.spines['left'].set_visible(True)
            ax.spines['left'].set_color('#E0E0E0') # Very faint gray 
            ax.spines['left'].set_linewidth(0.5)

    # --- 5. UNIFIED LEGEND ---
    l_g1 = mlines.Line2D([], [], color=colors[0], marker='o', linestyle='None', markersize=3)
    l_g2 = mlines.Line2D([], [], color=colors[1], marker='o', linestyle='None', markersize=3)
    
    # Place legend in the first panel, upper left
    #axes[0].legend(handles=[l_g1, l_g2], labels=[group_keys[0], group_keys[1]],
    #               frameon=False, loc='upper left', handletextpad=0.2, borderpad=0)

    # --- 6. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    
    # Slight w_pad ensures the text labels don't crash into the faint boundaries
    fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)